# Structured R1 — Semantic Branch Ablation Study

This notebook continues the sequential Structured R1 ablation analysis after the participation, local-temporal, and global-temporal studies.

The preceding ablations established the selected evidence configuration:

```text
Participation evidence:
    `speaks` only
    (`filtered_turns` removed)

Local temporal evidence:
    response-offset distribution only
    (overlap removed)

Global temporal evidence:
    complete global branch retained

Semantic evidence:
    coarse + focused summaries
```

This selected configuration achieved the strongest development performance observed at this stage:

| Configuration | NORMAL | LAG | WRONG | SILENT | Accuracy |
|---|---:|---:|---:|---:|---:|
| **Selected Structured R1 configuration** | **76/100** | **93/100** | **96/100** | **100/100** | **91.25%** |

The purpose of this notebook is to test the contribution of the **semantic branch as a whole** while keeping the selected participation and temporal evidence fixed.

---

# Ablation Design

## S0 — Remove the complete semantic branch

The full semantic payload is removed from the model-facing input.

This includes both semantic resolutions:

```text
coarse summaries
focused summaries
```

The Structured R1 output schema is adapted accordingly: `semantic_assessment` is removed because no semantic evidence remains available to assess.

The model therefore receives only:

```text
Participation:
    `speaks`

Local temporal:
    complete response-offset distribution

Global temporal:
    all five global alignment features

Semantic:
    removed
```

The following components remain unchanged:

- the same 400-case development set;
- the same Qwen2.5-Omni model;
- the same deterministic decoding settings;
- the same `speaks`-only participation representation;
- no `filtered_turns`;
- the same local response-offset evidence;
- no overlap evidence;
- the complete global temporal branch;
- the same frozen NORMAL temporal reference;
- the same Structured R1 decision policy, except for removing semantic-specific instructions and fields.

The ablation is therefore a controlled test of what happens when semantic evidence is removed while all selected non-semantic evidence is held fixed.

---

## Main Result

Removing the semantic branch produces:

| Configuration | NORMAL | LAG | WRONG | SILENT | Accuracy |
|---|---:|---:|---:|---:|---:|
| Selected setup with semantics | 76/100 | 93/100 | 96/100 | 100/100 | **91.25%** |
| **No semantic branch** | **47/100** | **99/100** | **99/100** | **100/100** | **86.25%** |

The overall accuracy decreases, but the class-specific behaviour changes in a highly informative way:

- NORMAL preservation collapses from **76/100 to 47/100**;
- LAG detection increases from **93/100 to 99/100**;
- Wrong Partner detection increases from **96/100 to 99/100**;
- Silent Partner remains **100/100**.

This is one of the clearest demonstrations of **cross-branch interference / functional coupling** in the unified reasoner.

The semantic branch is not acting only as an independent detector of semantic mismatch.

Instead, when semantic evidence is present and interpreted as conversationally compatible, it also appears to **moderate the interpretation of temporal deviations**, helping preserve genuine NORMAL conversations. Removing semantics leaves the temporal branches unchanged, yet the reasoner becomes substantially more anomaly-sensitive.

The effect is therefore not simply:

```text
remove semantic evidence
        ↓
worse Wrong Partner detection
```

In fact, Wrong Partner and LAG detection both improve after semantic removal.

The dominant change is instead:

```text
remove semantic compatibility evidence
        ↓
less calibration toward normality
        ↓
temporal deviations are interpreted more aggressively
        ↓
LAG / WRONG sensitivity increases
        ↓
NORMAL preservation collapses
```

This result shows that the Structured R1 evidence branches are **not functionally independent or simply additive**. Semantic evidence can change how the same temporal evidence is interpreted even when the temporal features themselves remain fixed.

For this reason, the semantic ablation becomes a central diagnostic result in the thesis discussion of evidence dominance and cross-branch coupling.

> **Reproducibility note:** every code cell, saved output, execution count, code-cell metadata field, manual prompt, prompt inspection, evaluation result, and diagnostic output is preserved exactly as executed. Only this introductory Markdown cell has been rewritten for the public repository.

## 1. Install dependencies

Run this cell once in a fresh Colab runtime. Then select:

`Runtime → Restart session`

and continue from the next cell.

In [ ]:
!pip uninstall -y transformers
!pip install -U "transformers>=4.57.0" accelerate bitsandbytes sentencepiece
!pip install -U qwen-omni-utils decord ffmpeg-python
!apt-get update -qq
!apt-get install -y -qq ffmpeg

Found existing installation: transformers 5.13.1
Uninstalling transformers-5.13.1:
  Successfully uninstalled transformers-5.13.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 102.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 146.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 40.0 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


## 2. Mount Google Drive and configure artifact paths

The folder name still contains `1_2_3sec` for historical reasons. The finalized 400-case database is loaded from the exact path below.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")


from pathlib import Path
from collections import Counter
import copy
import json

import pandas as pd
from IPython.display import display


OUT_DIR = Path(
    "/content/drive/MyDrive/"
    "qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec"
)


REFERENCE_BASE_STATS_PATH = (
    OUT_DIR
    / "frozen_reference_base_statistics.json"
)


REFERENCE_SHIFT_STATS_PATH = (
    OUT_DIR
    / "frozen_reference_global_shift_statistics.json"
)


FINAL_DATABASE_PATH = (
    OUT_DIR
    / (
        "consolidation_all_400_cases_with_"
        "temporal_and_semantic_summaries.json"
    )
)


MODEL_ID = "Qwen/Qwen2.5-Omni-7B"

MODEL_PATH = Path(
    "/content/drive/MyDrive/Qwen2.5-Omni-7B"
)


required_paths = {
    "Frozen base statistics": (
        REFERENCE_BASE_STATS_PATH
    ),

    "Frozen global-shift statistics": (
        REFERENCE_SHIFT_STATS_PATH
    ),

    "Final 400-case database": (
        FINAL_DATABASE_PATH
    ),

    "Local Qwen checkpoint": (
        MODEL_PATH
    ),
}


print("=" * 88)
print("ARTIFACT PATH AUDIT")
print("=" * 88)

for name, path in required_paths.items():

    print(
        f"{name}:",
        path,
    )

    print(
        "  exists:",
        path.exists(),
    )


assert OUT_DIR.exists(), (
    f"Project directory not found: {OUT_DIR}"
)

assert REFERENCE_BASE_STATS_PATH.exists(), (
    "Frozen base-statistics file not found: "
    f"{REFERENCE_BASE_STATS_PATH}"
)

assert REFERENCE_SHIFT_STATS_PATH.exists(), (
    "Frozen global-shift-statistics file not found: "
    f"{REFERENCE_SHIFT_STATS_PATH}"
)

assert FINAL_DATABASE_PATH.exists(), (
    "Final consolidation database not found: "
    f"{FINAL_DATABASE_PATH}"
)

assert MODEL_PATH.exists(), (
    "Local Qwen checkpoint not found: "
    f"{MODEL_PATH}"
)

Mounted at /content/drive
ARTIFACT PATH AUDIT
Frozen base statistics: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/frozen_reference_base_statistics.json
  exists: True
Frozen global-shift statistics: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/frozen_reference_global_shift_statistics.json
  exists: True
Final 400-case database: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_all_400_cases_with_temporal_and_semantic_summaries.json
  exists: True
Local Qwen checkpoint: /content/drive/MyDrive/Qwen2.5-Omni-7B
  exists: True


## 3. Load the frozen NORMAL reference statistics

Both frozen JSON files contain separate reference profiles. This experiment deliberately extracts and retains only:

```text
reference_profile = NORMAL
```

The lag reference profiles are not included in the binary-only experiment state at this stage.

In [ ]:
# ============================================================
# LOAD FROZEN STATISTICS AND RETAIN NORMAL ONLY
# ============================================================

def load_json_list(
    path,
):
    data = json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


    assert isinstance(
        data,
        list,
    ), (
        f"Expected a JSON list in {path}"
    )


    return data


frozen_base_records_all_profiles = (
    load_json_list(
        REFERENCE_BASE_STATS_PATH
    )
)


frozen_shift_records_all_profiles = (
    load_json_list(
        REFERENCE_SHIFT_STATS_PATH
    )
)


base_profiles = [
    str(
        record.get(
            "reference_profile"
        )
    )

    for record
    in frozen_base_records_all_profiles
]


shift_profiles = [
    str(
        record.get(
            "reference_profile"
        )
    )

    for record
    in frozen_shift_records_all_profiles
]


assert len(
    base_profiles
) == len(
    set(
        base_profiles
    )
)


assert len(
    shift_profiles
) == len(
    set(
        shift_profiles
    )
)


assert "NORMAL" in base_profiles
assert "NORMAL" in shift_profiles


normal_base_matches = [
    record

    for record
    in frozen_base_records_all_profiles

    if str(
        record.get(
            "reference_profile"
        )
    ) == "NORMAL"
]


normal_shift_matches = [
    record

    for record
    in frozen_shift_records_all_profiles

    if str(
        record.get(
            "reference_profile"
        )
    ) == "NORMAL"
]


assert len(
    normal_base_matches
) == 1


assert len(
    normal_shift_matches
) == 1


frozen_normal_base_statistics = copy.deepcopy(
    normal_base_matches[0]
)


frozen_normal_shift_statistics = copy.deepcopy(
    normal_shift_matches[0]
)


assert (
    frozen_normal_base_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


assert (
    frozen_normal_shift_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


# This is the only frozen reference object that should later
# be passed to the binary-only prompt constructor.
FROZEN_NORMAL_REFERENCE = {
    "base_statistics": (
        frozen_normal_base_statistics
    ),

    "global_shift_statistics": (
        frozen_normal_shift_statistics
    ),
}


normal_base_display_df = pd.DataFrame([
    {
        "metric": key,
        "value": value,
    }

    for key, value
    in frozen_normal_base_statistics.items()

    if key != "reference_profile"
])


normal_shift_display_df = pd.DataFrame([
    {
        "metric": key,
        "value": value,
    }

    for key, value
    in frozen_normal_shift_statistics.items()

    if key != "reference_profile"
])


print("=" * 88)
print("FROZEN REFERENCE FILES LOADED")
print("=" * 88)

print(
    "Profiles stored in base-statistics file:",
    base_profiles,
)

print(
    "Profiles stored in global-shift file:",
    shift_profiles,
)


print("\n" + "=" * 88)
print("FROZEN NORMAL BASE TEMPORAL STATISTICS")
print("=" * 88)

display(
    normal_base_display_df
)


print("\n" + "=" * 88)
print("FROZEN NORMAL GLOBAL-SHIFT STATISTICS")
print("=" * 88)

display(
    normal_shift_display_df
)


print(
    "\nOnly NORMAL frozen references retained:",
    list(
        FROZEN_NORMAL_REFERENCE.keys()
    ),
)

FROZEN REFERENCE FILES LOADED
Profiles stored in base-statistics file: ['NORMAL', 'LAG_1', 'LAG_2', 'LAG_3']
Profiles stored in global-shift file: ['NORMAL', 'LAG_1', 'LAG_2', 'LAG_3']

FROZEN NORMAL BASE TEMPORAL STATISTICS


,metric,value
0,num_original_A_turns,11.800000
1,num_original_B_turns,11.080000
2,num_removed_A_backchannels,1.740000
3,num_removed_B_backchannels,1.660000
4,num_filtered_A_turns,10.060000
5,num_filtered_B_turns,9.420000
6,num_offsets,5.760000
7,num_negative,2.040000
8,num_positive,3.720000
9,offset_mean,0.302128



FROZEN NORMAL GLOBAL-SHIFT STATISTICS


,metric,value
0,best_B_correction_shift_seconds__mean,-0.230000
1,best_B_correction_shift_seconds__median,-0.000000
2,estimated_B_lateness_seconds__mean,0.500000
3,estimated_B_lateness_seconds__median,0.000000
4,alignment_score_gain_vs_zero__mean,0.023376
5,alignment_score_gain_vs_zero__median,0.009105
6,best_num_bilateral_events__mean,11.940000
7,best_event_coverage__mean,0.595175



Only NORMAL frozen references retained: ['base_statistics', 'global_shift_statistics']


## 4. Load and audit the final 400-case database

This cell verifies:

- exactly 400 unique cases;
- exactly 100 cases per family;
- 100 `NORMAL` and 300 `ANOMALOUS` binary labels;
- participant-level `speaks` agrees with the final filtered VAD turns;
- every sample has all four coarse and all four focused summaries;
- no focused summary contains the legacy `speaks` field.

In [ ]:
# ============================================================
# LOAD FINAL DATABASE
# ============================================================

consolidation_cases = load_json_list(
    FINAL_DATABASE_PATH
)


assert len(
    consolidation_cases
) == 400


# ============================================================
# NORMALIZE CASE FAMILY
# ============================================================

def get_case_family(
    case,
):
    variant = str(
        case.get(
            "case_variant",
            "",
        )
    ).lower()


    if variant == "normal":
        return "normal"


    if variant == "wrong_partner":
        return "wrong_partner"


    if variant == "silent_partner":
        return "silent_partner"


    if variant.startswith(
        "lag"
    ):
        return "lag"


    raise ValueError(
        "Unknown case_variant: "
        f"{case.get('case_variant')}"
    )


# ============================================================
# GLOBAL COUNTS
# ============================================================

case_ids = [
    str(
        case[
            "case_id"
        ]
    )

    for case
    in consolidation_cases
]


assert len(
    case_ids
) == len(
    set(
        case_ids
    )
)


family_counts = Counter(
    get_case_family(
        case
    )

    for case
    in consolidation_cases
)


expected_family_counts = {
    "normal": 100,
    "wrong_partner": 100,
    "lag": 100,
    "silent_partner": 100,
}


assert dict(
    family_counts
) == expected_family_counts


binary_label_counts = Counter(
    str(
        case[
            "gold_binary_label"
        ]
    )

    for case
    in consolidation_cases
)


assert binary_label_counts[
    "NORMAL"
] == 100


assert binary_label_counts[
    "ANOMALOUS"
] == 300


lag_variant_counts = Counter(
    str(
        case.get(
            "case_variant"
        )
    )

    for case
    in consolidation_cases

    if get_case_family(
        case
    ) == "lag"
)


# ============================================================
# PARTICIPANT AND SEMANTIC AUDITS
# ============================================================

SEGMENT_NAMES = [
    "segment_0_0_to_60_seconds",
    "segment_1_60_to_120_seconds",
]


num_participant_records_checked = 0
num_semantic_slots_checked = 0


for case in consolidation_cases:

    for (
        role,
        turns_key,
    ) in [
        (
            "participant_A",
            "participant_A_filtered_turns",
        ),

        (
            "participant_B",
            "participant_B_filtered_turns",
        ),
    ]:

        participant = case[
            role
        ]


        turns = case[
            turns_key
        ]


        assert isinstance(
            participant,
            dict,
        )


        assert isinstance(
            turns,
            list,
        )


        assert (
            "speaks"
            in participant
        )


        assert isinstance(
            participant[
                "speaks"
            ],
            bool,
        )


        assert (
            participant[
                "speaks"
            ]
            ==
            (
                len(
                    turns
                ) > 0
            )
        ), (
            "Participant-level speaks disagrees with "
            f'VAD turns in {case["case_id"]} / {role}'
        )


        participant_semantics = (
            case[
                "semantic_summaries"
            ][role]
        )


        for segment_name in SEGMENT_NAMES:

            segment_record = (
                participant_semantics[
                    segment_name
                ]
            )


            coarse_summary = (
                segment_record.get(
                    "coarse_summary"
                )
            )


            focused_summary = (
                segment_record.get(
                    "focused_summary"
                )
            )


            assert isinstance(
                coarse_summary,
                dict,
            ), (
                "Missing coarse summary in "
                f'{case["case_id"]} / {role} / {segment_name}'
            )


            assert isinstance(
                focused_summary,
                dict,
            ), (
                "Missing focused summary in "
                f'{case["case_id"]} / {role} / {segment_name}'
            )


            assert (
                "speaks"
                not in focused_summary
            ), (
                "Legacy focused-summary speaks field found in "
                f'{case["case_id"]} / {role} / {segment_name}'
            )


            num_semantic_slots_checked += 1


        num_participant_records_checked += 1


# ============================================================
# DISPLAY DATABASE COMPOSITION
# ============================================================

family_count_df = pd.DataFrame([
    {
        "case_family": family,
        "num_cases": count,
    }

    for family, count
    in sorted(
        family_counts.items()
    )
])


binary_count_df = pd.DataFrame([
    {
        "gold_binary_label": label,
        "num_cases": count,
    }

    for label, count
    in sorted(
        binary_label_counts.items()
    )
])


lag_variant_df = pd.DataFrame([
    {
        "case_variant": variant,
        "num_cases": count,
    }

    for variant, count
    in sorted(
        lag_variant_counts.items()
    )
])


print("=" * 88)
print("FINAL CONSOLIDATION DATABASE AUDIT PASSED")
print("=" * 88)

print(
    "Database:",
    FINAL_DATABASE_PATH,
)

print(
    "Total unique cases:",
    len(
        consolidation_cases
    ),
)

print(
    "Participant records checked:",
    num_participant_records_checked,
)

print(
    "Semantic participant-segment slots checked:",
    num_semantic_slots_checked,
)


print("\nCASE FAMILIES")

display(
    family_count_df
)


print("\nBINARY LABELS")

display(
    binary_count_df
)


print("\nLAG VARIANTS STORED IN THE FINAL DATABASE")

display(
    lag_variant_df
)

FINAL CONSOLIDATION DATABASE AUDIT PASSED
Database: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/consolidation_all_400_cases_with_temporal_and_semantic_summaries.json
Total unique cases: 400
Participant records checked: 800
Semantic participant-segment slots checked: 1600

CASE FAMILIES


,case_family,num_cases
0,lag,100
1,normal,100
2,silent_partner,100
3,wrong_partner,100



BINARY LABELS


,gold_binary_label,num_cases
0,ANOMALOUS,300
1,NORMAL,100



LAG VARIANTS STORED IN THE FINAL DATABASE


,case_variant,num_cases
0,lag_2sec,50
1,lag_3sec,50


## 5. Participant-level `speaks` overview

The field is derived exclusively from each sample's final filtered VAD turns.

In [ ]:
# ============================================================
# PARTICIPANT-LEVEL SPEAKS SUMMARY
# ============================================================

speaks_rows = []


for case in consolidation_cases:

    speaks_rows.append({
        "case_family": (
            get_case_family(
                case
            )
        ),

        "case_id": (
            case[
                "case_id"
            ]
        ),

        "participant_A_speaks": (
            case[
                "participant_A"
            ][
                "speaks"
            ]
        ),

        "participant_B_speaks": (
            case[
                "participant_B"
            ][
                "speaks"
            ]
        ),

        "participant_A_num_turns": len(
            case[
                "participant_A_filtered_turns"
            ]
        ),

        "participant_B_num_turns": len(
            case[
                "participant_B_filtered_turns"
            ]
        ),
    })


speaks_df = pd.DataFrame(
    speaks_rows
)


speaks_summary_df = (
    speaks_df
    .groupby(
        "case_family",
        as_index=False,
    )
    .agg(
        total_cases=(
            "case_id",
            "size",
        ),

        participant_A_speaks_true=(
            "participant_A_speaks",
            "sum",
        ),

        participant_B_speaks_true=(
            "participant_B_speaks",
            "sum",
        ),
    )
)


speaks_summary_df[
    "participant_A_speaks_false"
] = (
    speaks_summary_df[
        "total_cases"
    ]
    -
    speaks_summary_df[
        "participant_A_speaks_true"
    ]
)


speaks_summary_df[
    "participant_B_speaks_false"
] = (
    speaks_summary_df[
        "total_cases"
    ]
    -
    speaks_summary_df[
        "participant_B_speaks_true"
    ]
)


speaks_summary_df = speaks_summary_df[
    [
        "case_family",
        "total_cases",

        "participant_A_speaks_true",
        "participant_A_speaks_false",

        "participant_B_speaks_true",
        "participant_B_speaks_false",
    ]
]


display(
    speaks_summary_df
)

,case_family,total_cases,participant_A_speaks_true,participant_A_speaks_false,participant_B_speaks_true,participant_B_speaks_false
0,lag,100,100,0,100,0
1,normal,100,100,0,100,0
2,silent_partner,100,100,0,0,100
3,wrong_partner,100,100,0,100,0


## 6. Interactive case inspection

The inspector supports all four families:

- `NORMAL`
- `WRONG_PARTNER`
- `LAG`
- `SILENT_PARTNER`

It can display participant metadata, VAD-derived `speaks`, filtered turns, temporal features, semantic summaries, and the complete sample JSON.

The direct function can also be used without widgets:

```python
inspect_case(
    family="normal",
    sample_index=0,
)
```

In [ ]:
# ============================================================
# INTERACTIVE DATABASE INSPECTOR
# ============================================================

import ipywidgets as widgets

from IPython.display import (
    clear_output,
    display,
)


try:

    from google.colab import output

    output.enable_custom_widget_manager()

except Exception:

    pass


# ============================================================
# SPLIT AND SORT CASES
# ============================================================

cases_by_family = {
    "normal": [],
    "wrong_partner": [],
    "lag": [],
    "silent_partner": [],
}


for case in consolidation_cases:

    cases_by_family[
        get_case_family(
            case
        )
    ].append(
        case
    )


for family in cases_by_family:

    cases_by_family[
        family
    ] = sorted(
        cases_by_family[
            family
        ],

        key=lambda case: str(
            case.get(
                "case_id",
                "",
            )
        ),
    )


# ============================================================
# DISPLAY HELPERS
# ============================================================

def participant_row(
    case,
    role,
):
    participant = case[
        role
    ]


    turns_key = (
        f"{role}_filtered_turns"
    )


    turns = case.get(
        turns_key,
        [],
    )


    return {
        "role": role,

        "conversation_id": (
            participant.get(
                "conversation_id"
            )
        ),

        "participant_id": (
            participant.get(
                "participant_id"
            )
        ),

        "speaks": (
            participant.get(
                "speaks"
            )
        ),

        "num_filtered_turns": len(
            turns
        ),

        "metadata_path": (
            participant.get(
                "metadata_path"
            )
        ),
    }


def display_semantics(
    case,
    role,
):
    print(
        "\n" + "-" * 88
    )

    print(
        f"{role.upper()} SEMANTIC SUMMARIES"
    )

    print(
        "-" * 88
    )


    participant_semantics = (
        case[
            "semantic_summaries"
        ][role]
    )


    for segment_name in SEGMENT_NAMES:

        segment_record = (
            participant_semantics[
                segment_name
            ]
        )


        print(
            f"\nSEGMENT: {segment_name}"
        )


        print(
            "\nCoarse summary:"
        )


        print(
            json.dumps(
                segment_record[
                    "coarse_summary"
                ],
                indent=2,
                ensure_ascii=False,
            )
        )


        print(
            "\nFocused summary:"
        )


        print(
            json.dumps(
                segment_record[
                    "focused_summary"
                ],
                indent=2,
                ensure_ascii=False,
            )
        )


# ============================================================
# MAIN INSPECTION FUNCTION
# ============================================================

def inspect_case(
    family,
    sample_index,
    show_turns=True,
    show_temporal_features=True,
    show_semantics=True,
    show_full_json=False,
):
    assert family in cases_by_family


    family_cases = cases_by_family[
        family
    ]


    sample_index = int(
        sample_index
    )


    assert (
        0
        <= sample_index
        < len(
            family_cases
        )
    )


    case = family_cases[
        sample_index
    ]


    print("=" * 88)
    print("CONSOLIDATION CASE INSPECTION")
    print("=" * 88)

    print(
        "Family:",
        family,
    )

    print(
        "Family position:",
        f"{sample_index + 1}/{len(family_cases)}",
    )

    print(
        "case_id:",
        case.get(
            "case_id"
        ),
    )

    print(
        "source_group_id:",
        case.get(
            "source_group_id"
        ),
    )

    print(
        "case_variant:",
        case.get(
            "case_variant"
        ),
    )

    print(
        "gold_binary_label:",
        case.get(
            "gold_binary_label"
        ),
    )

    print(
        "gold_anomaly_type:",
        case.get(
            "gold_anomaly_type"
        ),
    )

    print(
        "A_source_conversation:",
        case.get(
            "A_source_conversation"
        ),
    )

    print(
        "B_source_conversation:",
        case.get(
            "B_source_conversation"
        ),
    )


    print(
        "\nPARTICIPANTS"
    )


    display(
        pd.DataFrame([
            participant_row(
                case,
                "participant_A",
            ),

            participant_row(
                case,
                "participant_B",
            ),
        ])
    )


    print(
        "\nSEMANTIC COVERAGE"
    )


    print(
        json.dumps(
            case.get(
                "semantic_summary_coverage"
            ),
            indent=2,
            ensure_ascii=False,
        )
    )


    if show_turns:

        print(
            "\n" + "=" * 88
        )

        print(
            "PARTICIPANT A FILTERED TURNS"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case.get(
                    "participant_A_filtered_turns"
                ),
                indent=2,
                ensure_ascii=False,
            )
        )


        print(
            "\n" + "=" * 88
        )

        print(
            "PARTICIPANT B FILTERED TURNS"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case.get(
                    "participant_B_filtered_turns"
                ),
                indent=2,
                ensure_ascii=False,
            )
        )


    if show_temporal_features:

        print(
            "\n" + "=" * 88
        )

        print(
            "LOCAL TEMPORAL FEATURES"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case.get(
                    "local_temporal_features"
                ),
                indent=2,
                ensure_ascii=False,
            )
        )


        print(
            "\n" + "=" * 88
        )

        print(
            "GLOBAL SHIFT FEATURES"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case.get(
                    "global_shift_features"
                ),
                indent=2,
                ensure_ascii=False,
            )
        )


    if show_semantics:

        display_semantics(
            case,
            "participant_A",
        )


        display_semantics(
            case,
            "participant_B",
        )


    if show_full_json:

        print(
            "\n" + "=" * 88
        )

        print(
            "COMPLETE SAMPLE JSON"
        )

        print(
            "=" * 88
        )


        print(
            json.dumps(
                case,
                indent=2,
                ensure_ascii=False,
            )
        )


    return case


# ============================================================
# WIDGETS
# ============================================================

family_labels = {
    "normal": "NORMAL",
    "wrong_partner": "WRONG_PARTNER",
    "lag": "LAG",
    "silent_partner": "SILENT_PARTNER",
}


family_dropdown = widgets.Dropdown(
    options=[
        (
            family_labels[
                family
            ],
            family,
        )

        for family in [
            "normal",
            "wrong_partner",
            "lag",
            "silent_partner",
        ]
    ],

    value="normal",

    description="Family:",

    style={
        "description_width": "initial"
    },

    layout=widgets.Layout(
        width="450px"
    ),
)


case_dropdown = widgets.Dropdown(
    description="Sample:",

    style={
        "description_width": "initial"
    },

    layout=widgets.Layout(
        width="900px"
    ),
)


show_turns_checkbox = widgets.Checkbox(
    value=True,
    description="Show filtered turns",
    indent=False,
)


show_temporal_checkbox = widgets.Checkbox(
    value=True,
    description="Show temporal features",
    indent=False,
)


show_semantics_checkbox = widgets.Checkbox(
    value=True,
    description="Show semantic summaries",
    indent=False,
)


show_full_json_checkbox = widgets.Checkbox(
    value=False,
    description="Show complete JSON",
    indent=False,
)


inspector_output = widgets.Output()


def refresh_case_options(
    *args,
):
    family = family_dropdown.value


    family_cases = cases_by_family[
        family
    ]


    case_dropdown.options = [
        (
            (
                f"{index + 1:03d}/100 | "
                f'{case.get("case_id")} | '
                f'{case.get("case_variant")}'
            ),

            index,
        )

        for index, case
        in enumerate(
            family_cases
        )
    ]


    case_dropdown.value = 0


def render_case(
    *args,
):
    if case_dropdown.value is None:
        return


    with inspector_output:

        clear_output(
            wait=True
        )


        inspect_case(
            family=(
                family_dropdown.value
            ),

            sample_index=(
                case_dropdown.value
            ),

            show_turns=(
                show_turns_checkbox.value
            ),

            show_temporal_features=(
                show_temporal_checkbox.value
            ),

            show_semantics=(
                show_semantics_checkbox.value
            ),

            show_full_json=(
                show_full_json_checkbox.value
            ),
        )


family_dropdown.observe(
    refresh_case_options,
    names="value",
)


family_dropdown.observe(
    render_case,
    names="value",
)


case_dropdown.observe(
    render_case,
    names="value",
)


show_turns_checkbox.observe(
    render_case,
    names="value",
)


show_temporal_checkbox.observe(
    render_case,
    names="value",
)


show_semantics_checkbox.observe(
    render_case,
    names="value",
)


show_full_json_checkbox.observe(
    render_case,
    names="value",
)


refresh_case_options()


controls = widgets.VBox([
    family_dropdown,
    case_dropdown,

    widgets.HBox([
        show_turns_checkbox,
        show_temporal_checkbox,
    ]),

    widgets.HBox([
        show_semantics_checkbox,
        show_full_json_checkbox,
    ]),
])


print("=" * 88)
print("INTERACTIVE INSPECTOR READY")
print("=" * 88)

for family in [
    "normal",
    "wrong_partner",
    "lag",
    "silent_partner",
]:

    print(
        family,
        len(
            cases_by_family[
                family
            ]
        ),
    )


display(
    controls,
    inspector_output,
)


render_case()


INTERACTIVE INSPECTOR READY
normal 100
wrong_partner 100
lag 100
silent_partner 100


Output()

## 7. Load Qwen2.5-Omni Thinker


In [ ]:
import torch
from transformers import Qwen2_5OmniThinkerForConditionalGeneration, Qwen2_5OmniProcessor
from qwen_omni_utils import process_mm_info

MODEL_ID = "Qwen/Qwen2.5-Omni-7B"

model = Qwen2_5OmniThinkerForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
)

processor = Qwen2_5OmniProcessor.from_pretrained(MODEL_ID)

print("Loaded:", MODEL_ID)

config.json:   0%|          | 0.00/13.2k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/233k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1346 [00:00<?, ?it/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/832 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-Omni-7B


# Ηelper Functions For Qwen Calls

In [ ]:
def extract_json_from_text(text: str):
    text = text.strip()

    # remove markdown fences
    text = re.sub(r"^```(?:json)?", "", text.strip(), flags=re.IGNORECASE)
    text = re.sub(r"```$", "", text.strip())

    try:
        return json.loads(text)
    except Exception:
        pass

    # find first balanced JSON object
    start = text.find("{")
    if start == -1:
        return {"parse_error": True, "raw_output": text}

    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                candidate = text[start:i+1]
                try:
                    return json.loads(candidate)
                except Exception:
                    return {
                        "parse_error": True,
                        "raw_output": text,
                        "json_candidate": candidate,
                    }

    return {"parse_error": True, "raw_output": text}

def qwen_video_text(video_path: str, prompt: str, max_new_tokens: int = 350):
    messages = [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "You are Qwen, a virtual human developed by the Qwen Team, "
                        "Alibaba Group, capable of perceiving auditory and visual inputs, "
                        "as well as generating text and speech."
                    )
                }
            ],
        },
        {
            "role": "user",
            "content": [
                {"type": "video", "video": video_path, "fps": 7.0},
                {"type": "text", "text": prompt},
            ],
        },
    ]

    text = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False
    )

    audios, images, videos = process_mm_info(
        messages,
        use_audio_in_video=True
    )

    inputs = processor(
        text=text,
        audio=audios,
        images=images,
        videos=videos,
        return_tensors="pt",
        padding=True,
        use_audio_in_video=True,
    )

    inputs = inputs.to(model.device)

    print("Input tokens:", inputs["input_ids"].shape[1])

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    generated = processor.batch_decode(
        output_ids[:, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return generated

def qwen_text_only(prompt: str, max_new_tokens: int = 700):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        processor_kwargs={
            "padding": True,
        },
    )

    inputs = {
        k: v.to(model.device) if hasattr(v, "to") else v
        for k, v in inputs.items()
    }

    input_token_count = int(
        inputs["input_ids"].shape[-1]
    )


    print(
        "Input tokens:",
        input_token_count,
    )


    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    generated = processor.batch_decode(
        output_ids[:, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return generated


# Shared frozen NORMAL reference text

This is the same reference-text construction used by the original binary experiments. No anomaly-specific profile is added here; R2 and R3 load their exact saved prompts, which already contain the frozen LAG₂/LAG₃ profile text.

In [ ]:
# ============================================================
# BUILD AND PRINT THE SELECTED NORMAL-ONLY REFERENCE TEXT
#
# Only the frozen NORMAL profile is used.
#
# Included base metrics:
#   - num_offsets
#   - offset_mean
#   - offset_median
#   - offset_max
#   - offset_p75
#   - offset_p90
#   - percent_above_1_5
#   - clean_overlap_seconds
#
# Included global metrics:
#   - best_B_correction_shift_seconds__mean
#   - best_B_correction_shift_seconds__median
#   - estimated_B_lateness_seconds__mean
#   - estimated_B_lateness_seconds__median
#   - alignment_score_gain_vs_zero__mean
#   - alignment_score_gain_vs_zero__median
#   - best_num_bilateral_events__mean
#   - best_event_coverage__mean
#
# No other frozen statistics are exposed.
# ============================================================

import json
import math
import numbers


# ============================================================
# STRICT NORMAL PROFILE AUDIT
# ============================================================

assert isinstance(
    frozen_normal_base_statistics,
    dict,
)


assert isinstance(
    frozen_normal_shift_statistics,
    dict,
)


assert (
    frozen_normal_base_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


assert (
    frozen_normal_shift_statistics[
        "reference_profile"
    ]
    == "NORMAL"
)


# ============================================================
# NUMERIC VALIDATION HELPER
# ============================================================

def require_finite_numeric(
    record,
    field_name,
):
    """
    Read one required numeric field and verify
    that it exists and contains a finite number.
    """

    assert field_name in record, (
        f"Missing required field: {field_name}"
    )


    value = record[
        field_name
    ]


    assert isinstance(
        value,
        numbers.Real,
    ) and not isinstance(
        value,
        bool,
    ), (
        f"Expected numeric value for {field_name}, "
        f"found {type(value).__name__}: {value}"
    )


    value = float(
        value
    )


    assert math.isfinite(
        value
    ), (
        f"Non-finite value for {field_name}: {value}"
    )


    return value


def normalize_negative_zero(
    value,
):
    """
    Convert -0.0 to 0.0 for cleaner prompt text.
    """

    if abs(
        value
    ) < 1e-12:

        return 0.0


    return value


# ============================================================
# SELECT ONLY THE REQUIRED NORMAL BASE STATISTICS
# ============================================================

normal_num_offsets = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "num_offsets",
    )
)


normal_offset_mean = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_mean",
    )
)


normal_offset_median = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_median",
    )
)


normal_offset_max = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_max",
    )
)


normal_offset_p75 = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_p75",
    )
)


normal_offset_p90 = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "offset_p90",
    )
)


normal_percent_above_1_5 = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "percent_above_1_5",
    )
)


normal_clean_overlap_seconds = (
    require_finite_numeric(
        frozen_normal_base_statistics,
        "clean_overlap_seconds",
    )
)


# ============================================================
# SELECT ONLY THE REQUIRED NORMAL GLOBAL-SHIFT STATISTICS
#
# IMPORTANT:
# These fields use a double underscore before
# "mean" and "median".
# ============================================================

normal_best_shift_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_B_correction_shift_seconds__mean",
    )
)


normal_best_shift_median = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_B_correction_shift_seconds__median",
    )
)


normal_lateness_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "estimated_B_lateness_seconds__mean",
    )
)


normal_lateness_median = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "estimated_B_lateness_seconds__median",
    )
)


normal_alignment_gain_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "alignment_score_gain_vs_zero__mean",
    )
)


normal_alignment_gain_median = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "alignment_score_gain_vs_zero__median",
    )
)


normal_bilateral_events_mean = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_num_bilateral_events__mean",
    )
)


normal_event_coverage_fraction = (
    require_finite_numeric(
        frozen_normal_shift_statistics,
        "best_event_coverage__mean",
    )
)


# ============================================================
# CLEAN DISPLAY VALUES
# ============================================================

normal_best_shift_mean = (
    normalize_negative_zero(
        normal_best_shift_mean
    )
)


normal_best_shift_median = (
    normalize_negative_zero(
        normal_best_shift_median
    )
)


normal_lateness_mean = (
    normalize_negative_zero(
        normal_lateness_mean
    )
)


normal_lateness_median = (
    normalize_negative_zero(
        normal_lateness_median
    )
)


normal_event_coverage_percent = (
    100.0
    * normal_event_coverage_fraction
)


# ============================================================
# STORE THE EXACT VALUES USED IN THE PROMPT
# ============================================================

normal_base_reference_values = {
    "num_signed_offsets": (
        normal_num_offsets
    ),

    "mean_signed_offset_seconds": (
        normal_offset_mean
    ),

    "median_signed_offset_seconds": (
        normal_offset_median
    ),

    "maximum_signed_offset_seconds": (
        normal_offset_max
    ),

    "p75_signed_offset_seconds": (
        normal_offset_p75
    ),

    "p90_signed_offset_seconds": (
        normal_offset_p90
    ),

    "percent_offsets_above_1_5_seconds": (
        normal_percent_above_1_5
    ),

    "filtered_clean_overlap_seconds": (
        normal_clean_overlap_seconds
    ),
}


normal_global_reference_values = {
    "mean_best_B_correction_shift_seconds": (
        normal_best_shift_mean
    ),

    "median_best_B_correction_shift_seconds": (
        normal_best_shift_median
    ),

    "mean_estimated_B_lateness_seconds": (
        normal_lateness_mean
    ),

    "median_estimated_B_lateness_seconds": (
        normal_lateness_median
    ),

    "mean_alignment_score_gain_vs_zero": (
        normal_alignment_gain_mean
    ),

    "median_alignment_score_gain_vs_zero": (
        normal_alignment_gain_median
    ),

    "mean_num_bilateral_alignment_events": (
        normal_bilateral_events_mean
    ),

    "mean_bilateral_event_coverage_percent": (
        normal_event_coverage_percent
    ),
}


assert len(
    normal_base_reference_values
) == 8


assert len(
    normal_global_reference_values
) == 8


# ============================================================
# BUILD NORMAL LOCAL-TEMPORAL REFERENCE TEXT
# ============================================================

normal_base_reference_text = f"""
Frozen NORMAL local temporal reference statistics:

- Number of signed offsets is around {normal_num_offsets:.2f}.
- Mean signed offset is around {normal_offset_mean:.2f} seconds.
- Median signed offset is around {normal_offset_median:.2f} seconds.
- Maximum signed offset is around {normal_offset_max:.2f} seconds.
- P75 signed offset is around {normal_offset_p75:.2f} seconds.
- P90 signed offset is around {normal_offset_p90:.2f} seconds.
- Approximately {normal_percent_above_1_5:.1f}% of offsets are above 1.5 seconds.
- Filtered clean overlap is around {normal_clean_overlap_seconds:.2f} seconds.

These values were calculated only from the frozen NORMAL reference conversations.
They are soft reference patterns and must not be treated as hard classification thresholds.
""".strip()


# ============================================================
# BUILD NORMAL GLOBAL-ALIGNMENT REFERENCE TEXT
# ============================================================

normal_global_reference_text = f"""
Frozen NORMAL global alignment-shift reference statistics:

- Mean best B correction shift is around {normal_best_shift_mean:.2f} seconds.
- Median best B correction shift is around {normal_best_shift_median:.2f} seconds.
- Mean estimated B lateness is around {normal_lateness_mean:.2f} seconds.
- Median estimated B lateness is around {normal_lateness_median:.2f} seconds.
- Mean alignment score gain versus zero shift is around {normal_alignment_gain_mean:.3f}.
- Median alignment score gain versus zero shift is around {normal_alignment_gain_median:.3f}.
- Mean number of bilateral alignment events is around {normal_bilateral_events_mean:.2f}.
- Mean bilateral event coverage is around {normal_event_coverage_percent:.2f}%.

These values were calculated only from the frozen NORMAL reference conversations.
They describe typical global-alignment behavior in the frozen NORMAL set and are not hard classification thresholds.
""".strip()


# ============================================================
# PRINT BOTH PROMPT SECTIONS
# ============================================================

print("=" * 88)
print("normal_base_reference_text")
print("=" * 88)

print(
    normal_base_reference_text
)


print("\n" + "=" * 88)
print("normal_global_reference_text")
print("=" * 88)

print(
    normal_global_reference_text
)


# ============================================================
# PRINT THE EXACT STRUCTURED VALUES USED
# ============================================================

print("\n" + "=" * 88)
print("EXACT SELECTED NORMAL BASE VALUES")
print("=" * 88)

print(
    json.dumps(
        normal_base_reference_values,
        indent=2,
        ensure_ascii=False,
    )
)


print("\n" + "=" * 88)
print("EXACT SELECTED NORMAL GLOBAL VALUES")
print("=" * 88)

print(
    json.dumps(
        normal_global_reference_values,
        indent=2,
        ensure_ascii=False,
    )
)


# ============================================================
# STRICT CONTENT AUDIT
# ============================================================

combined_reference_text = (
    normal_base_reference_text
    + "\n"
    + normal_global_reference_text
)


# No anomaly-specific labels or profiles.
for forbidden_text in [
    "LAG +1",
    "LAG +2",
    "LAG +3",
    "wrong_partner",
    "silent_partner",
    "anomaly_type",
]:

    assert (
        forbidden_text.lower()
        not in combined_reference_text.lower()
    )


# Unwanted frozen reference fields must not appear.
for excluded_text in [
    "original A turns",
    "original B turns",
    "removed A backchannels",
    "removed B backchannels",
    "filtered A turns",
    "filtered B turns",
    "number of negative",
    "number of positive",
    "minimum signed offset",
    "clean overlap percentage",
]:

    assert (
        excluded_text.lower()
        not in combined_reference_text.lower()
    )


# All required selected metrics must appear.
for required_text in [
    "Number of signed offsets",
    "Mean signed offset",
    "Median signed offset",
    "Maximum signed offset",
    "P75 signed offset",
    "P90 signed offset",
    "offsets are above 1.5 seconds",
    "Filtered clean overlap",
    "Mean best B correction shift",
    "Median best B correction shift",
    "Mean estimated B lateness",
    "Median estimated B lateness",
    "Mean alignment score gain",
    "Median alignment score gain",
    "Mean number of bilateral alignment events",
    "Mean bilateral event coverage",
]:

    assert (
        required_text
        in combined_reference_text
    )


print("\n" + "=" * 88)
print("SELECTED NORMAL-ONLY REFERENCE TEXT READY")
print("=" * 88)

print(
    "Base reference metrics included:",
    len(
        normal_base_reference_values
    ),
)

print(
    "Global reference metrics included:",
    len(
        normal_global_reference_values
    ),
)

print(
    "No additional frozen statistics were exposed."
)

print(
    "No explicit anomaly profile or anomaly type was included."
)

normal_base_reference_text
Frozen NORMAL local temporal reference statistics:

- Number of signed offsets is around 5.76.
- Mean signed offset is around 0.30 seconds.
- Median signed offset is around 0.26 seconds.
- Maximum signed offset is around 1.05 seconds.
- P75 signed offset is around 0.59 seconds.
- P90 signed offset is around 0.83 seconds.
- Approximately 6.7% of offsets are above 1.5 seconds.
- Filtered clean overlap is around 6.25 seconds.

These values were calculated only from the frozen NORMAL reference conversations.
They are soft reference patterns and must not be treated as hard classification thresholds.

normal_global_reference_text
Frozen NORMAL global alignment-shift reference statistics:

- Mean best B correction shift is around -0.23 seconds.
- Median best B correction shift is around 0.00 seconds.
- Mean estimated B lateness is around 0.50 seconds.
- Median estimated B lateness is around 0.00 seconds.
- Mean alignment score gain versus zero shift is around 0.023.

# Shared original Structured R1 projection and evaluation helpers

The cell below remains the same source projection used by the previous notebooks. It can build the complete original payload internally.

The later semantic-ablation helper creates a deep-copied model-facing view that removes:

- participant filtered-turn lists,
- local overlap features,
- the complete coarse and focused semantic payload.

It retains:

- participant `speaks`,
- all local offset-distribution features,
- all global temporal features.


In [ ]:

# ============================================================
# SHARED STRUCTURED-REASONING EXPERIMENT HELPERS
#
# The complete original Structured R1 projection is retained here
# as an internal source projection. The manual semantic-ablation helper defined later creates a deep-copied
# model-facing view that removes filtered turns, overlap and semantic summaries,
# retains participant-level `speaks`, and preserves local offsets and global features.
#
# The three source prompts are loaded from their exact saved
# prompt_template.txt files.
#
# The only prompt change is replacement of the original binary
# OUTPUT block with one common structured-reasoning JSON schema.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import copy
import hashlib
import json
import re
import time

import pandas as pd
import torch

from tqdm.auto import tqdm


import difflib

MAX_NEW_TOKENS_REASONING = 256
MAX_NEW_TOKENS = MAX_NEW_TOKENS_REASONING

LABELS = [
    "NORMAL",
    "ANOMALOUS",
]


# ============================================================
# EXACT SEMANTIC FIELDS PASSED TO THE MODEL
# ============================================================

COARSE_FIELDS = [
    "speech_content_summary",
    "apparent_topic",
]


FOCUSED_FIELDS = [
    "detailed_speech_summary",
    "main_topic",
    "secondary_topics",
    "key_semantic_details",
    "summary_specificity",
    "unclear_content",
    "confidence",
]


SEGMENT_MAP = {
    "segment_0_0_to_60_seconds": (
        "segment_0"
    ),

    "segment_1_60_to_120_seconds": (
        "segment_1"
    ),
}


# ============================================================
# EXACT TEMPORAL FIELDS PASSED TO THE MODEL
# ============================================================

LOCAL_FEATURE_FIELDS = [
    "clean_overlap_seconds",
    "clean_overlap_percent",

    "signed_strict_offsets_seconds",
    "num_signed_strict_offsets",

    "offset_mean_seconds",
    "offset_median_seconds",
    "offset_max_seconds",
    "offset_p75_seconds",
    "offset_p90_seconds",

    "num_offsets_above_1_5_seconds",
    "percent_offsets_above_1_5_seconds",
]


GLOBAL_FEATURE_FIELDS = [
    "best_B_correction_shift_seconds",
    "estimated_B_lateness_seconds",
    "alignment_score_gain_vs_zero",
    "best_num_bilateral_events",
    "best_event_coverage",
]


# ============================================================
# FIELDS THAT MUST NEVER APPEAR IN THE MODEL PROMPT
# ============================================================

FORBIDDEN_PROMPT_KEYS = [
    "case_id",
    "source_group_id",
    "pair_index",

    "gold_binary_label",
    "gold_anomaly_type",

    "case_variant",
    "pairing_type",

    "conversation_id",
    "participant_id",
    "metadata_path",

    "num_raw_vad_entries",

    "A_source_conversation",
    "B_source_conversation",

    "semantic_summary_source",
    "semantic_summary_coverage",
]


# ============================================================
# GENERAL HELPERS
# ============================================================

def canonical_json(
    value,
):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    )


def sha256_text(
    text,
):
    return hashlib.sha256(
        text.encode(
            "utf-8"
        )
    ).hexdigest()


def select_exact_fields(
    record,
    fields,
    context,
):
    assert isinstance(
        record,
        dict,
    ), (
        f"Expected dictionary for {context}"
    )


    missing_fields = [
        field

        for field in fields

        if field not in record
    ]


    assert not missing_fields, (
        f"Missing fields in {context}: "
        f"{missing_fields}"
    )


    return {
        field: copy.deepcopy(
            record[
                field
            ]
        )

        for field in fields
    }


# ============================================================
# TURN PROJECTION
#
# Database format:
#   {"start": 1.2, "end": 3.4}
#
# Prompt format:
#   [1.2, 3.4]
# ============================================================

def compact_turns(
    turns,
    context,
):
    assert isinstance(
        turns,
        list,
    ), (
        f"Expected list for {context}"
    )


    compact = []


    for index, turn in enumerate(
        turns
    ):

        assert isinstance(
            turn,
            dict,
        ), (
            f"Invalid turn in {context} "
            f"at index {index}"
        )


        assert "start" in turn
        assert "end" in turn


        start = float(
            turn[
                "start"
            ]
        )


        end = float(
            turn[
                "end"
            ]
        )


        assert end >= start


        compact.append([
            start,
            end,
        ])


    return compact


# ============================================================
# SEMANTIC INPUT PROJECTION
#
# Participant IDs and other metadata are deliberately excluded.
# ============================================================

def build_semantic_input(
    case,
):
    semantic_input = {}


    for role in [
        "participant_A",
        "participant_B",
    ]:

        participant_semantics = (
            case[
                "semantic_summaries"
            ][role]
        )


        role_output = {}


        for (
            database_segment_name,
            prompt_segment_name,
        ) in SEGMENT_MAP.items():

            segment_record = (
                participant_semantics[
                    database_segment_name
                ]
            )


            coarse_summary = (
                select_exact_fields(
                    segment_record[
                        "coarse_summary"
                    ],

                    COARSE_FIELDS,

                    (
                        f"{role}/"
                        f"{database_segment_name}/"
                        "coarse_summary"
                    ),
                )
            )


            focused_summary = (
                select_exact_fields(
                    segment_record[
                        "focused_summary"
                    ],

                    FOCUSED_FIELDS,

                    (
                        f"{role}/"
                        f"{database_segment_name}/"
                        "focused_summary"
                    ),
                )
            )


            # Legacy focused-summary speaks must not exist.
            assert (
                "speaks"
                not in focused_summary
            )


            role_output[
                prompt_segment_name
            ] = {
                "coarse_summary": (
                    coarse_summary
                ),

                "focused_summary": (
                    focused_summary
                ),
            }


        semantic_input[
            role
        ] = role_output


    return semantic_input


# ============================================================
# BUILD THE EXACT MODEL INPUT
# ============================================================

def build_binary_model_input(
    case,
):
    local_features = (
        select_exact_fields(
            case[
                "local_temporal_features"
            ],

            LOCAL_FEATURE_FIELDS,

            "local_temporal_features",
        )
    )


    global_features_raw = (
        select_exact_fields(
            case[
                "global_shift_features"
            ],

            GLOBAL_FEATURE_FIELDS,

            "global_shift_features",
        )
    )


    # Stored in the database as a fraction.
    # Presented in the prompt as a percentage so that it is
    # directly comparable with the frozen NORMAL percentage.
    event_coverage_fraction = float(
        global_features_raw.pop(
            "best_event_coverage"
        )
    )


    global_features = {
        **global_features_raw,

        "best_event_coverage_percent": round(
            100.0
            * event_coverage_fraction,

            6,
        ),
    }


    payload = {
        "analysis_duration_seconds": float(
            case[
                "analysis_duration_seconds"
            ]
        ),

        "participant_A": {
            "speaks": bool(
                case[
                    "participant_A"
                ][
                    "speaks"
                ]
            ),

            "filtered_turns": compact_turns(
                case[
                    "participant_A_filtered_turns"
                ],

                "participant_A_filtered_turns",
            ),
        },

        "participant_B": {
            "speaks": bool(
                case[
                    "participant_B"
                ][
                    "speaks"
                ]
            ),

            "filtered_turns": compact_turns(
                case[
                    "participant_B_filtered_turns"
                ],

                "participant_B_filtered_turns",
            ),
        },

        "local_temporal_features": (
            local_features
        ),

        "global_shift_features": (
            global_features
        ),

        "semantic_summaries": (
            build_semantic_input(
                case
            )
        ),
    }


    assert set(
        payload
    ) == {
        "analysis_duration_seconds",
        "participant_A",
        "participant_B",
        "local_temporal_features",
        "global_shift_features",
        "semantic_summaries",
    }


    return payload


# ============================================================
# BUILD THE FINAL PROMPT FOR ONE CASE
# ============================================================

def build_binary_prompt_from_template(
    case,
    prompt_template,
):
    payload = build_binary_model_input(
        case
    )


    prompt = (
        prompt_template
        .format(
            normal_base_reference_text=(
                normal_base_reference_text
            ),

            normal_global_reference_text=(
                normal_global_reference_text
            ),

            duration_seconds=(
                payload[
                    "analysis_duration_seconds"
                ]
            ),

            participant_A_speaks=(
                canonical_json(
                    payload[
                        "participant_A"
                    ][
                        "speaks"
                    ]
                )
            ),

            participant_A_turns=(
                canonical_json(
                    payload[
                        "participant_A"
                    ][
                        "filtered_turns"
                    ]
                )
            ),

            participant_B_speaks=(
                canonical_json(
                    payload[
                        "participant_B"
                    ][
                        "speaks"
                    ]
                )
            ),

            participant_B_turns=(
                canonical_json(
                    payload[
                        "participant_B"
                    ][
                        "filtered_turns"
                    ]
                )
            ),

            local_temporal_features=(
                json.dumps(
                    payload[
                        "local_temporal_features"
                    ],
                    indent=2,
                    ensure_ascii=False,
                )
            ),

            global_shift_features=(
                json.dumps(
                    payload[
                        "global_shift_features"
                    ],
                    indent=2,
                    ensure_ascii=False,
                )
            ),

            semantic_summaries=(
                json.dumps(
                    payload[
                        "semantic_summaries"
                    ],
                    indent=2,
                    ensure_ascii=False,
                )
            ),
        )
    )


    # Strict leakage audit.
    prompt_lower = prompt.lower()


    for forbidden_key in (
        FORBIDDEN_PROMPT_KEYS
    ):

        assert (
            forbidden_key.lower()
            not in prompt_lower
        ), (
            "Forbidden field name leaked into prompt: "
            f"{forbidden_key}"
        )


    return (
        prompt,
        payload,
    )


# ============================================================
# TEXT-ONLY QWEN INFERENCE
#
# The existing qwen_text_only helper used 700 output tokens.
# Here only 64 are needed because the expected output is:
#
#   {"label": "NORMAL"}
#
# or:
#
#   {"label": "ANOMALOUS"}
# ============================================================

def qwen_text_only_binary(
    prompt,
    max_new_tokens=MAX_NEW_TOKENS,
):
    messages = [
        {
            "role": "user",

            "content": [
                {
                    "type": "text",
                    "text": prompt,
                },
            ],
        }
    ]


    inputs = (
        processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",

            processor_kwargs={
                "padding": True,
            },
        )
    )


    inputs = {
        key: (
            value.to(
                model.device
            )
            if hasattr(
                value,
                "to",
            )
            else value
        )

        for key, value
        in inputs.items()
    }


    input_token_count = int(
        inputs[
            "input_ids"
        ].shape[-1]
    )


    with torch.inference_mode():

        output_ids = model.generate(
            **inputs,

            max_new_tokens=(
                max_new_tokens
            ),

            do_sample=False,

            use_cache=True,
        )


    generated_ids = output_ids[
        :,
        inputs[
            "input_ids"
        ].shape[1]:,
    ]


    raw_output = (
        processor.batch_decode(
            generated_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )[0]
    )


    return (
        raw_output,
        input_token_count,
    )


# ============================================================
# ROBUST JSON PARSING
# ============================================================

def extract_first_json_object(
    text,
):
    cleaned = text.strip()


    cleaned = re.sub(
        r"^```(?:json)?\s*",
        "",
        cleaned,
        flags=re.IGNORECASE,
    )


    cleaned = re.sub(
        r"\s*```$",
        "",
        cleaned,
    )


    try:

        return json.loads(
            cleaned
        )

    except Exception:

        pass


    decoder = json.JSONDecoder()


    possible_starts = [
        index

        for index, character
        in enumerate(
            cleaned
        )

        if character == "{"
    ]


    for start in possible_starts:

        try:

            parsed, _ = (
                decoder.raw_decode(
                    cleaned[
                        start:
                    ]
                )
            )


            return parsed

        except Exception:

            continue


    return None

# ============================================================
# STRUCTURED REASONING SCHEMA
# ============================================================

REASONING_SCHEMA_KEYS = [
    "participation_assessment",
    "local_temporal_assessment",
    "global_temporal_assessment",
    "temporal_assessment",
    "semantic_assessment",
    "decisive_dimension",
    "label",
]


REASONING_ALLOWED_VALUES = {
    "participation_assessment": {
        "VALID",
        "INVALID",
    },

    "local_temporal_assessment": {
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    },

    "global_temporal_assessment": {
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    },

    "temporal_assessment": {
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    },

    "semantic_assessment": {
        "COMPATIBLE",
        "INCOMPATIBLE",
        "LIMITED",
    },

    "decisive_dimension": {
        "PARTICIPATION",
        "TEMPORAL",
        "SEMANTIC",
        "NONE",
    },

    "label": {
        "NORMAL",
        "ANOMALOUS",
    },
}


OLD_BINARY_OUTPUT_BLOCK = """
============================================================
OUTPUT
============================================================

Return ONLY one valid JSON object with exactly this schema:

{{
  "label": "NORMAL or ANOMALOUS"
}}

Do not include confidence.
Do not include reasoning.
Do not include an anomaly type.
Do not include Markdown.
Do not include any text outside the JSON object.
""".strip()


STRUCTURED_REASONING_OUTPUT_BLOCK = """
============================================================
STRUCTURED REASONING OUTPUT
============================================================

Preserve all evidence definitions, comparison rules, decision criteria,
and the final binary decision policy stated above.

Before selecting the final label, expose the following fixed structured
assessments.

These fields are diagnostic outputs only.

They must not introduce new evidence, new thresholds, a new voting rule,
or any change to the existing decision policy.

Use the available evidence exactly as instructed above.

Assessment meanings:

- participation_assessment:
  - VALID
  - INVALID

- local_temporal_assessment:
  - NORMAL
  - ANOMALOUS
  - LIMITED

- global_temporal_assessment:
  - NORMAL
  - ANOMALOUS
  - LIMITED

- temporal_assessment:
  - NORMAL
  - ANOMALOUS
  - LIMITED

- semantic_assessment:
  - COMPATIBLE
  - INCOMPATIBLE
  - LIMITED

- decisive_dimension:
  - PARTICIPATION
  - TEMPORAL
  - SEMANTIC
  - NONE

The local and global temporal assessments must reflect the reliability
rules already defined in the prompt.

The combined temporal assessment must be based on the existing temporal
decision policy.

The final label must follow the existing final decision policy exactly.

Do not infer or output an anomaly subtype.
Do not infer or output a delay magnitude.
Do not provide free-form reasoning.

============================================================
OUTPUT
============================================================

Return ONLY one valid JSON object with exactly this schema:

{{
  "participation_assessment": "VALID or INVALID",
  "local_temporal_assessment": "NORMAL or ANOMALOUS or LIMITED",
  "global_temporal_assessment": "NORMAL or ANOMALOUS or LIMITED",
  "temporal_assessment": "NORMAL or ANOMALOUS or LIMITED",
  "semantic_assessment": "COMPATIBLE or INCOMPATIBLE or LIMITED",
  "decisive_dimension": "PARTICIPATION or TEMPORAL or SEMANTIC or NONE",
  "label": "NORMAL or ANOMALOUS"
}}

Do not include confidence.
Do not include an anomaly type.
Do not include Markdown.
Do not include any text outside the JSON object.
""".strip()


def build_structured_reasoning_prompt_template(
    source_prompt_template,
):
    assert isinstance(
        source_prompt_template,
        str,
    )


    assert source_prompt_template.endswith(
        OLD_BINARY_OUTPUT_BLOCK
    ), (
        "The saved source prompt does not end with the exact "
        "original binary OUTPUT block."
    )


    reasoning_prompt_template = (
        source_prompt_template[
            :-len(
                OLD_BINARY_OUTPUT_BLOCK
            )
        ]
        + STRUCTURED_REASONING_OUTPUT_BLOCK
    )


    reconstructed_source_prompt = (
        reasoning_prompt_template[
            :-len(
                STRUCTURED_REASONING_OUTPUT_BLOCK
            )
        ]
        + OLD_BINARY_OUTPUT_BLOCK
    )


    assert (
        reconstructed_source_prompt
        == source_prompt_template
    ), (
        "Unexpected source-prompt modification."
    )


    return reasoning_prompt_template


def parse_structured_reasoning_prediction(
    raw_output,
):
    parsed = extract_first_json_object(
        raw_output
    )


    normalized = {
        key: None
        for key in REASONING_SCHEMA_KEYS
    }


    schema_errors = []


    if isinstance(
        parsed,
        dict,
    ):

        for key in REASONING_SCHEMA_KEYS:

            if key in parsed:

                normalized[
                    key
                ] = str(
                    parsed[
                        key
                    ]
                ).strip().upper()


        for key in REASONING_SCHEMA_KEYS:

            if normalized[
                key
            ] not in REASONING_ALLOWED_VALUES[
                key
            ]:

                schema_errors.append(
                    (
                        f"{key}: "
                        f"{normalized[key]}"
                    )
                )


        exact_keys = (
            set(
                parsed.keys()
            )
            == set(
                REASONING_SCHEMA_KEYS
            )
        )


        schema_exact = (
            exact_keys
            and
            not schema_errors
        )


        label = normalized[
            "label"
        ]


        if label in LABELS:

            return {
                "prediction": label,

                "parse_mode": (
                    "structured_json"
                ),

                "schema_exact": bool(
                    schema_exact
                ),

                "parsed_output": parsed,

                "schema_errors": (
                    schema_errors
                ),

                **{
                    key: normalized[
                        key
                    ]

                    for key in (
                        REASONING_SCHEMA_KEYS
                    )

                    if key != "label"
                },
            }


    plain_output = (
        raw_output
        .strip()
        .strip('"')
        .strip("'")
        .upper()
    )


    if plain_output in LABELS:

        return {
            "prediction": plain_output,

            "parse_mode": (
                "exact_plaintext_fallback"
            ),

            "schema_exact": False,

            "parsed_output": None,

            "schema_errors": [
                "Structured reasoning fields missing."
            ],

            **{
                key: None

                for key in (
                    REASONING_SCHEMA_KEYS
                )

                if key != "label"
            },
        }


    return {
        "prediction": None,

        "parse_mode": "invalid",

        "schema_exact": False,

        "parsed_output": parsed,

        "schema_errors": (
            schema_errors
            if schema_errors
            else [
                "No valid structured JSON prediction."
            ]
        ),

        **{
            key: normalized[
                key
            ]

            for key in (
                REASONING_SCHEMA_KEYS
            )

            if key != "label"
        },
    }


# ============================================================
# FULL-SEMANTICS PAYLOAD AUDIT
# ============================================================

def collect_nested_keys_reasoning(
    value,
):
    keys = set()


    if isinstance(
        value,
        dict,
    ):

        for key, nested_value in value.items():

            keys.add(
                str(key)
            )


            keys.update(
                collect_nested_keys_reasoning(
                    nested_value
                )
            )


    elif isinstance(
        value,
        list,
    ):

        for item in value:

            keys.update(
                collect_nested_keys_reasoning(
                    item
                )
            )


    return keys


FULL_SEMANTIC_REQUIRED_KEYS = [
    "semantic_summaries",
    "coarse_summary",
    "focused_summary",

    "speech_content_summary",
    "apparent_topic",

    "detailed_speech_summary",
    "main_topic",
    "secondary_topics",
    "key_semantic_details",
    "summary_specificity",
    "unclear_content",
    "confidence",
]


# ============================================================
# EXACT PROMPT CONFIGURATION
# ============================================================

def prepare_reasoning_experiment(
    *,
    experiment_version,
    source_experiment_name,
    experiment_title,
    required_source_markers,
    forbidden_source_markers,
    temporal_profiles_used,
    assessment_policy,
):
    source_experiment_dir = (
        OUT_DIR
        / source_experiment_name
    )


    source_prompt_path = (
        source_experiment_dir
        / "prompt_template.txt"
    )


    assert source_prompt_path.exists(), (
        "Saved source prompt not found:\n"
        f"{source_prompt_path}"
    )


    source_prompt_template = (
        source_prompt_path.read_text(
            encoding="utf-8"
        )
    )


    for marker in required_source_markers:

        assert marker in source_prompt_template, (
            "Required source-prompt marker missing: "
            f"{marker}"
        )


    for marker in forbidden_source_markers:

        assert marker not in source_prompt_template, (
            "Forbidden source-prompt marker found: "
            f"{marker}"
        )


    for semantic_marker in [
        "Coarse semantic information:",
        "Focused semantic information:",
        "- detailed_speech_summary",
        "- main_topic",
        "- secondary_topics",
        "- key_semantic_details",
    ]:

        assert semantic_marker in source_prompt_template, (
            "The source prompt is not a full-semantics prompt. "
            f"Missing: {semantic_marker}"
        )


    reasoning_prompt_template = (
        build_structured_reasoning_prompt_template(
            source_prompt_template
        )
    )


    experiment_dir = (
        OUT_DIR
        / experiment_version
    )


    experiment_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    paths = {
        "prediction_cache": (
            experiment_dir
            / "predictions_cache.json"
        ),

        "predictions_csv": (
            experiment_dir
            / "predictions_all_400.csv"
        ),

        "metrics_json": (
            experiment_dir
            / "metrics.json"
        ),

        "confusion_matrix_csv": (
            experiment_dir
            / "confusion_matrix.csv"
        ),

        "family_metrics_csv": (
            experiment_dir
            / "metrics_by_case_family.csv"
        ),

        "variant_metrics_csv": (
            experiment_dir
            / "metrics_by_case_variant.csv"
        ),

        "errors_csv": (
            experiment_dir
            / "classification_errors.csv"
        ),

        "reasoning_assessments_csv": (
            experiment_dir
            / "reasoning_assessments.csv"
        ),

        "reasoning_inconsistencies_csv": (
            experiment_dir
            / "reasoning_inconsistencies.csv"
        ),

        "assessment_distributions_csv": (
            experiment_dir
            / "assessment_distributions.csv"
        ),

        "prompt_template": (
            experiment_dir
            / "prompt_template.txt"
        ),

        "source_prompt_copy": (
            experiment_dir
            / "source_prompt_template.txt"
        ),

        "prompt_diff": (
            experiment_dir
            / "prompt_diff_vs_source.txt"
        ),

        "experiment_manifest": (
            experiment_dir
            / "experiment_manifest.json"
        ),
    }


    source_prompt_sha256 = sha256_text(
        source_prompt_template
    )


    reasoning_prompt_sha256 = sha256_text(
        reasoning_prompt_template
    )


    reasoning_schema_sha256 = sha256_text(
        STRUCTURED_REASONING_OUTPUT_BLOCK
    )


    normal_reference_sha256 = sha256_text(
        normal_base_reference_text
        + "\n"
        + normal_global_reference_text
    )


    paths[
        "prompt_template"
    ].write_text(
        reasoning_prompt_template,
        encoding="utf-8",
    )


    paths[
        "source_prompt_copy"
    ].write_text(
        source_prompt_template,
        encoding="utf-8",
    )


    prompt_diff_lines = difflib.unified_diff(
        source_prompt_template.splitlines(),
        reasoning_prompt_template.splitlines(),

        fromfile=(
            f"{source_experiment_name}/prompt_template.txt"
        ),

        tofile=(
            f"{experiment_version}/prompt_template.txt"
        ),

        lineterm="",
    )


    paths[
        "prompt_diff"
    ].write_text(
        "\n".join(
            prompt_diff_lines
        ),
        encoding="utf-8",
    )


    example_prompt, example_payload = (
        build_binary_prompt_from_template(
            consolidation_cases[0],
            reasoning_prompt_template,
        )
    )


    payload_keys = collect_nested_keys_reasoning(
        example_payload
    )


    for required_key in FULL_SEMANTIC_REQUIRED_KEYS:

        assert required_key in payload_keys, (
            "Full-semantic input field missing: "
            f"{required_key}"
        )


    assert (
        "STRUCTURED REASONING OUTPUT"
        in example_prompt
    )


    assert (
        "Do not include reasoning."
        not in reasoning_prompt_template[
            -len(
                STRUCTURED_REASONING_OUTPUT_BLOCK
            ):
        ]
    )


    manifest = {
        "experiment_version": (
            experiment_version
        ),

        "experiment_title": (
            experiment_title
        ),

        "source_experiment": (
            source_experiment_name
        ),

        "source_prompt_path": str(
            source_prompt_path
        ),

        "source_prompt_sha256": (
            source_prompt_sha256
        ),

        "reasoning_prompt_sha256": (
            reasoning_prompt_sha256
        ),

        "reasoning_schema_sha256": (
            reasoning_schema_sha256
        ),

        "normal_reference_sha256": (
            normal_reference_sha256
        ),

        "semantic_input": (
            "coarse_and_focused"
        ),

        "focused_summaries_used": True,

        "temporal_profiles_used": (
            temporal_profiles_used
        ),

        "assessment_policy": (
            assessment_policy
        ),

        "max_new_tokens": (
            MAX_NEW_TOKENS_REASONING
        ),

        "model_id": (
            MODEL_ID
        ),

        "decoding": {
            "do_sample": False,
            "use_cache": True,
        },

        "only_prompt_change": (
            "The exact original binary OUTPUT block was "
            "replaced by the common structured-reasoning "
            "OUTPUT block."
        ),

        "all_model_facing_evidence_unchanged": (
            True
        ),

        "all_pre_output_prompt_text_unchanged": (
            True
        ),
    }


    paths[
        "experiment_manifest"
    ].write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


    config = {
        **manifest,

        "experiment_dir": (
            experiment_dir
        ),

        "source_prompt_template": (
            source_prompt_template
        ),

        "reasoning_prompt_template": (
            reasoning_prompt_template
        ),

        "paths": paths,
    }


    print("=" * 88)
    print(
        f"{experiment_title} — CONFIGURATION READY"
    )
    print("=" * 88)

    print(
        "Experiment version:",
        experiment_version,
    )

    print(
        "Source experiment:",
        source_experiment_name,
    )

    print(
        "Source prompt SHA256:",
        source_prompt_sha256,
    )

    print(
        "Reasoning prompt SHA256:",
        reasoning_prompt_sha256,
    )

    print(
        "Semantic input:",
        "coarse_and_focused",
    )

    print(
        "Focused summaries used:",
        True,
    )

    print(
        "Temporal profiles:",
        temporal_profiles_used,
    )

    print(
        "Assessment policy:",
        assessment_policy,
    )

    print(
        "Only source-prompt change:",
        (
            "binary OUTPUT block -> "
            "structured reasoning OUTPUT block"
        ),
    )

    print(
        "Example prompt characters:",
        len(
            example_prompt
        ),
    )

    print(
        "Prediction cache:",
        paths[
            "prediction_cache"
        ],
    )

    print(
        "Existing cache:",
        paths[
            "prediction_cache"
        ].exists(),
    )


    return config


# ============================================================
# CHECKPOINTED INFERENCE
# ============================================================

def reasoning_utc_now():
    return datetime.now(
        timezone.utc
    ).isoformat()


def reasoning_atomic_write_json(
    path,
    data,
):
    path = Path(
        path
    )


    temporary_path = path.with_suffix(
        path.suffix
        + ".tmp"
    )


    temporary_path.write_text(
        json.dumps(
            data,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


    temporary_path.replace(
        path
    )


def create_reasoning_cache(
    config,
):
    return {
        "experiment_version": (
            config[
                "experiment_version"
            ]
        ),

        "experiment_title": (
            config[
                "experiment_title"
            ]
        ),

        "source_experiment": (
            config[
                "source_experiment"
            ]
        ),

        "model_id": (
            MODEL_ID
        ),

        "source_prompt_sha256": (
            config[
                "source_prompt_sha256"
            ]
        ),

        "reasoning_prompt_sha256": (
            config[
                "reasoning_prompt_sha256"
            ]
        ),

        "reasoning_schema_sha256": (
            config[
                "reasoning_schema_sha256"
            ]
        ),

        "normal_reference_sha256": (
            config[
                "normal_reference_sha256"
            ]
        ),

        "semantic_input": (
            "coarse_and_focused"
        ),

        "focused_summaries_used": True,

        "temporal_profiles_used": (
            config[
                "temporal_profiles_used"
            ]
        ),

        "assessment_policy": (
            config[
                "assessment_policy"
            ]
        ),

        "max_new_tokens": (
            MAX_NEW_TOKENS_REASONING
        ),

        "created_at_utc": (
            reasoning_utc_now()
        ),

        "updated_at_utc": (
            reasoning_utc_now()
        ),

        "records": {},
    }


def run_reasoning_experiment(
    config,
):
    prediction_cache_path = (
        config[
            "paths"
        ][
            "prediction_cache"
        ]
    )


    if prediction_cache_path.exists():

        prediction_cache = json.loads(
            prediction_cache_path.read_text(
                encoding="utf-8"
            )
        )


        for key in [
            "experiment_version",
            "source_experiment",
            "model_id",
            "source_prompt_sha256",
            "reasoning_prompt_sha256",
            "reasoning_schema_sha256",
            "normal_reference_sha256",
            "semantic_input",
            "focused_summaries_used",
            "temporal_profiles_used",
            "assessment_policy",
            "max_new_tokens",
        ]:

            expected_value = (
                create_reasoning_cache(
                    config
                )[
                    key
                ]
            )


            assert (
                prediction_cache[
                    key
                ]
                == expected_value
            ), (
                f"Cache mismatch for {key}"
            )


        print(
            "Resuming cache:",
            prediction_cache_path,
        )

        print(
            "Existing records:",
            len(
                prediction_cache[
                    "records"
                ]
            ),
        )


    else:

        prediction_cache = (
            create_reasoning_cache(
                config
            )
        )


        reasoning_atomic_write_json(
            prediction_cache_path,
            prediction_cache,
        )


        print(
            "Created cache:",
            prediction_cache_path,
        )


    ordered_cases = sorted(
        consolidation_cases,

        key=lambda case: str(
            case[
                "case_id"
            ]
        ),
    )


    assert len(
        ordered_cases
    ) == 400


    for case in tqdm(
        ordered_cases,
        desc=(
            config[
                "experiment_version"
            ]
        ),
    ):

        case_id = str(
            case[
                "case_id"
            ]
        )


        prompt, model_input_payload = (
            build_binary_prompt_from_template(
                case,
                config[
                    "reasoning_prompt_template"
                ],
            )
        )


        current_payload_keys = (
            collect_nested_keys_reasoning(
                model_input_payload
            )
        )


        for required_key in (
            FULL_SEMANTIC_REQUIRED_KEYS
        ):

            assert required_key in (
                current_payload_keys
            ), (
                f"Missing full-semantic field "
                f"for case {case_id}: "
                f"{required_key}"
            )


        prompt_sha256 = sha256_text(
            prompt
        )


        input_payload_sha256 = sha256_text(
            canonical_json(
                model_input_payload
            )
        )


        existing_record = (
            prediction_cache[
                "records"
            ].get(
                case_id
            )
        )


        if (
            existing_record is not None
            and
            existing_record.get(
                "prediction"
            ) in LABELS
        ):

            assert (
                existing_record[
                    "prompt_sha256"
                ]
                == prompt_sha256
            )


            assert (
                existing_record[
                    "input_payload_sha256"
                ]
                == input_payload_sha256
            )


            continue


        started = time.perf_counter()


        try:

            raw_output, input_token_count = (
                qwen_text_only_binary(
                    prompt,
                    max_new_tokens=(
                        MAX_NEW_TOKENS_REASONING
                    ),
                )
            )


            parsed_result = (
                parse_structured_reasoning_prediction(
                    raw_output
                )
            )


            generation_error = None


        except Exception as exc:

            raw_output = ""


            input_token_count = None


            parsed_result = {
                "prediction": None,
                "parse_mode": "generation_error",
                "schema_exact": False,
                "parsed_output": None,
                "schema_errors": [],
                "participation_assessment": None,
                "local_temporal_assessment": None,
                "global_temporal_assessment": None,
                "temporal_assessment": None,
                "semantic_assessment": None,
                "decisive_dimension": None,
            }


            generation_error = (
                f"{type(exc).__name__}: {exc}"
            )


            if torch.cuda.is_available():

                torch.cuda.empty_cache()


        elapsed_seconds = (
            time.perf_counter()
            - started
        )


        prediction_cache[
            "records"
        ][case_id] = {
            "case_id": (
                case_id
            ),

            "source_group_id": str(
                case[
                    "source_group_id"
                ]
            ),

            "case_family": (
                get_case_family(
                    case
                )
            ),

            "case_variant": str(
                case[
                    "case_variant"
                ]
            ),

            "gold_binary_label": str(
                case[
                    "gold_binary_label"
                ]
            ).upper(),

            "prompt_sha256": (
                prompt_sha256
            ),

            "input_payload_sha256": (
                input_payload_sha256
            ),

            "semantic_input": (
                "coarse_and_focused"
            ),

            "focused_summaries_used": True,

            "input_token_count": (
                input_token_count
            ),

            "max_new_tokens": (
                MAX_NEW_TOKENS_REASONING
            ),

            "raw_output": (
                raw_output
            ),

            "prediction": (
                parsed_result[
                    "prediction"
                ]
            ),

            "participation_assessment": (
                parsed_result[
                    "participation_assessment"
                ]
            ),

            "local_temporal_assessment": (
                parsed_result[
                    "local_temporal_assessment"
                ]
            ),

            "global_temporal_assessment": (
                parsed_result[
                    "global_temporal_assessment"
                ]
            ),

            "temporal_assessment": (
                parsed_result[
                    "temporal_assessment"
                ]
            ),

            "semantic_assessment": (
                parsed_result[
                    "semantic_assessment"
                ]
            ),

            "decisive_dimension": (
                parsed_result[
                    "decisive_dimension"
                ]
            ),

            "parse_mode": (
                parsed_result[
                    "parse_mode"
                ]
            ),

            "schema_exact": bool(
                parsed_result[
                    "schema_exact"
                ]
            ),

            "schema_errors": (
                parsed_result[
                    "schema_errors"
                ]
            ),

            "parsed_output": (
                parsed_result[
                    "parsed_output"
                ]
            ),

            "generation_error": (
                generation_error
            ),

            "elapsed_seconds": round(
                elapsed_seconds,
                4,
            ),

            "completed_at_utc": (
                reasoning_utc_now()
            ),
        }


        prediction_cache[
            "updated_at_utc"
        ] = reasoning_utc_now()


        reasoning_atomic_write_json(
            prediction_cache_path,
            prediction_cache,
        )


    print("\n" + "=" * 88)
    print(
        f"{config['experiment_title']} — INFERENCE COMPLETE"
    )
    print("=" * 88)

    print(
        "Cached records:",
        len(
            prediction_cache[
                "records"
            ]
        ),
    )

    print(
        "Prediction cache:",
        prediction_cache_path,
    )


    return prediction_cache


# ============================================================
# EVALUATION
# ============================================================

def summarize_reasoning_group(
    group,
):
    valid_group = group[
        group[
            "valid_prediction"
        ]
    ]


    return {
        "total_cases": int(
            len(
                group
            )
        ),

        "valid_predictions": int(
            len(
                valid_group
            )
        ),

        "invalid_predictions": int(
            len(
                group
            )
            -
            len(
                valid_group
            )
        ),

        "predicted_NORMAL": int(
            (
                valid_group[
                    "prediction"
                ]
                == "NORMAL"
            ).sum()
        ),

        "predicted_ANOMALOUS": int(
            (
                valid_group[
                    "prediction"
                ]
                == "ANOMALOUS"
            ).sum()
        ),

        "correct_predictions": int(
            group[
                "correct"
            ].sum()
        ),

        "accuracy_on_valid": (
            float(
                (
                    valid_group[
                        "gold_label"
                    ]
                    ==
                    valid_group[
                        "prediction"
                    ]
                ).mean()
            )
            if len(
                valid_group
            )
            else float(
                "nan"
            )
        ),

        "strict_accuracy_invalid_as_wrong": float(
            group[
                "correct"
            ].mean()
        ),

        "exact_schema_rate": float(
            group[
                "schema_exact"
            ].mean()
        ),
    }


def evaluate_reasoning_experiment(
    config,
):
    from sklearn.metrics import (
        accuracy_score,
        balanced_accuracy_score,
        classification_report,
        confusion_matrix,
        f1_score,
        matthews_corrcoef,
        precision_score,
        recall_score,
    )

    from IPython.display import display


    prediction_cache = json.loads(
        config[
            "paths"
        ][
            "prediction_cache"
        ].read_text(
            encoding="utf-8"
        )
    )


    assert (
        prediction_cache[
            "experiment_version"
        ]
        == config[
            "experiment_version"
        ]
    )


    assert (
        prediction_cache[
            "reasoning_prompt_sha256"
        ]
        == config[
            "reasoning_prompt_sha256"
        ]
    )


    result_rows = []


    for case in consolidation_cases:

        case_id = str(
            case[
                "case_id"
            ]
        )


        record = (
            prediction_cache[
                "records"
            ].get(
                case_id
            )
        )


        result_rows.append({
            "case_id": (
                case_id
            ),

            "source_group_id": str(
                case[
                    "source_group_id"
                ]
            ),

            "case_family": (
                get_case_family(
                    case
                )
            ),

            "case_variant": str(
                case[
                    "case_variant"
                ]
            ),

            "gold_label": str(
                case[
                    "gold_binary_label"
                ]
            ).upper(),

            "prediction": (
                None
                if record is None
                else record.get(
                    "prediction"
                )
            ),

            "participation_assessment": (
                None
                if record is None
                else record.get(
                    "participation_assessment"
                )
            ),

            "local_temporal_assessment": (
                None
                if record is None
                else record.get(
                    "local_temporal_assessment"
                )
            ),

            "global_temporal_assessment": (
                None
                if record is None
                else record.get(
                    "global_temporal_assessment"
                )
            ),

            "temporal_assessment": (
                None
                if record is None
                else record.get(
                    "temporal_assessment"
                )
            ),

            "semantic_assessment": (
                None
                if record is None
                else record.get(
                    "semantic_assessment"
                )
            ),

            "decisive_dimension": (
                None
                if record is None
                else record.get(
                    "decisive_dimension"
                )
            ),

            "parse_mode": (
                "missing"
                if record is None
                else record.get(
                    "parse_mode"
                )
            ),

            "schema_exact": (
                False
                if record is None
                else bool(
                    record.get(
                        "schema_exact",
                        False,
                    )
                )
            ),

            "schema_errors": (
                None
                if record is None
                else json.dumps(
                    record.get(
                        "schema_errors",
                        [],
                    ),
                    ensure_ascii=False,
                )
            ),

            "input_token_count": (
                None
                if record is None
                else record.get(
                    "input_token_count"
                )
            ),

            "elapsed_seconds": (
                None
                if record is None
                else record.get(
                    "elapsed_seconds"
                )
            ),

            "raw_output": (
                ""
                if record is None
                else record.get(
                    "raw_output",
                    "",
                )
            ),

            "generation_error": (
                None
                if record is None
                else record.get(
                    "generation_error"
                )
            ),
        })


    results_df = pd.DataFrame(
        result_rows
    )


    results_df[
        "valid_prediction"
    ] = results_df[
        "prediction"
    ].isin(
        LABELS
    )


    results_df[
        "correct"
    ] = (
        results_df[
            "valid_prediction"
        ]
        &
        (
            results_df[
                "gold_label"
            ]
            ==
            results_df[
                "prediction"
            ]
        )
    )


    results_df[
        "normal_with_invalid_participation"
    ] = (
        (
            results_df[
                "prediction"
            ]
            == "NORMAL"
        )
        &
        (
            results_df[
                "participation_assessment"
            ]
            == "INVALID"
        )
    )


    results_df[
        "normal_with_anomalous_temporal"
    ] = (
        (
            results_df[
                "prediction"
            ]
            == "NORMAL"
        )
        &
        (
            results_df[
                "temporal_assessment"
            ]
            == "ANOMALOUS"
        )
    )


    results_df[
        "normal_with_incompatible_semantics"
    ] = (
        (
            results_df[
                "prediction"
            ]
            == "NORMAL"
        )
        &
        (
            results_df[
                "semantic_assessment"
            ]
            == "INCOMPATIBLE"
        )
    )


    results_df[
        "reasoning_inconsistency"
    ] = (
        results_df[
            [
                "normal_with_invalid_participation",
                "normal_with_anomalous_temporal",
                "normal_with_incompatible_semantics",
            ]
        ].any(
            axis=1
        )
    )


    assert len(
        results_df
    ) == 400


    assert (
        results_df[
            "case_id"
        ].is_unique
    )


    valid_df = results_df[
        results_df[
            "valid_prediction"
        ]
    ].copy()


    invalid_df = results_df[
        ~results_df[
            "valid_prediction"
        ]
    ].copy()


    assert len(
        valid_df
    ) > 0


    y_true = valid_df[
        "gold_label"
    ]


    y_pred = valid_df[
        "prediction"
    ]


    confusion = confusion_matrix(
        y_true,
        y_pred,
        labels=[
            "NORMAL",
            "ANOMALOUS",
        ],
    )


    confusion_df = pd.DataFrame(
        confusion,

        index=[
            "Gold NORMAL",
            "Gold ANOMALOUS",
        ],

        columns=[
            "Pred NORMAL",
            "Pred ANOMALOUS",
        ],
    )


    true_normal = int(
        confusion[
            0,
            0,
        ]
    )


    false_anomalous = int(
        confusion[
            0,
            1,
        ]
    )


    normal_recall = (
        true_normal
        /
        (
            true_normal
            + false_anomalous
        )
        if (
            true_normal
            + false_anomalous
        )
        else float(
            "nan"
        )
    )


    metrics = {
        "experiment_version": (
            config[
                "experiment_version"
            ]
        ),

        "source_experiment": (
            config[
                "source_experiment"
            ]
        ),

        "semantic_input": (
            "coarse_and_focused"
        ),

        "focused_summaries_used": True,

        "temporal_profiles_used": (
            config[
                "temporal_profiles_used"
            ]
        ),

        "assessment_policy": (
            config[
                "assessment_policy"
            ]
        ),

        "total_cases": int(
            len(
                results_df
            )
        ),

        "valid_predictions": int(
            len(
                valid_df
            )
        ),

        "invalid_predictions": int(
            len(
                invalid_df
            )
        ),

        "exact_json_schema_rate": float(
            results_df[
                "schema_exact"
            ].mean()
        ),

        "reasoning_inconsistency_count": int(
            results_df[
                "reasoning_inconsistency"
            ].sum()
        ),

        "accuracy_valid_predictions": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),

        "strict_accuracy_invalid_as_wrong": float(
            results_df[
                "correct"
            ].mean()
        ),

        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                y_pred,
            )
        ),

        "anomalous_precision": float(
            precision_score(
                y_true,
                y_pred,
                pos_label="ANOMALOUS",
                zero_division=0,
            )
        ),

        "anomalous_recall": float(
            recall_score(
                y_true,
                y_pred,
                pos_label="ANOMALOUS",
                zero_division=0,
            )
        ),

        "anomalous_f1": float(
            f1_score(
                y_true,
                y_pred,
                pos_label="ANOMALOUS",
                zero_division=0,
            )
        ),

        "normal_recall_specificity": float(
            normal_recall
        ),

        "matthews_correlation_coefficient": float(
            matthews_corrcoef(
                y_true,
                y_pred,
            )
        ),

        "confusion_matrix_label_order": [
            "NORMAL",
            "ANOMALOUS",
        ],

        "confusion_matrix": (
            confusion.tolist()
        ),
    }


    family_rows = []


    for (
        family_name,
        family_group,
    ) in results_df.groupby(
        "case_family",
        sort=True,
    ):

        family_rows.append({
            "case_family": (
                family_name
            ),

            **summarize_reasoning_group(
                family_group
            ),
        })


    family_metrics_df = pd.DataFrame(
        family_rows
    )


    variant_rows = []


    for (
        variant_name,
        variant_group,
    ) in results_df.groupby(
        "case_variant",
        sort=True,
    ):

        variant_rows.append({
            "case_variant": (
                variant_name
            ),

            **summarize_reasoning_group(
                variant_group
            ),
        })


    variant_metrics_df = pd.DataFrame(
        variant_rows
    )


    classification_report_df = pd.DataFrame(
        classification_report(
            y_true,
            y_pred,
            labels=[
                "NORMAL",
                "ANOMALOUS",
            ],
            output_dict=True,
            zero_division=0,
        )
    ).T


    error_df = results_df[
        (
            ~results_df[
                "valid_prediction"
            ]
        )
        |
        (
            results_df[
                "gold_label"
            ] != results_df[
                "prediction"
            ]
        )
    ].copy()


    reasoning_inconsistency_df = (
        results_df[
            results_df[
                "reasoning_inconsistency"
            ]
        ].copy()
    )


    assessment_rows = []


    for assessment_field in [
        "participation_assessment",
        "local_temporal_assessment",
        "global_temporal_assessment",
        "temporal_assessment",
        "semantic_assessment",
        "decisive_dimension",
    ]:

        counts = (
            results_df[
                assessment_field
            ]
            .fillna(
                "MISSING"
            )
            .value_counts(
                dropna=False
            )
        )


        for value, count in counts.items():

            assessment_rows.append({
                "assessment_field": (
                    assessment_field
                ),

                "assessment_value": (
                    value
                ),

                "count": int(
                    count
                ),
            })


    assessment_distributions_df = (
        pd.DataFrame(
            assessment_rows
        )
    )


    source_group_rows = []


    for (
        source_group_id,
        source_group,
    ) in results_df.groupby(
        "source_group_id"
    ):

        source_group_rows.append({
            "source_group_id": (
                source_group_id
            ),

            "num_cases": int(
                len(
                    source_group
                )
            ),

            "all_predictions_valid": bool(
                source_group[
                    "valid_prediction"
                ].all()
            ),

            "all_cases_correct": bool(
                source_group[
                    "valid_prediction"
                ].all()
                and
                source_group[
                    "correct"
                ].all()
            ),
        })


    source_group_df = pd.DataFrame(
        source_group_rows
    )


    assert len(
        source_group_df
    ) == 100


    assert set(
        source_group_df[
            "num_cases"
        ]
    ) == {
        4
    }


    metrics[
        "source_group_exact_match_rate"
    ] = float(
        source_group_df[
            "all_cases_correct"
        ].mean()
    )


    paths = config[
        "paths"
    ]


    results_df.to_csv(
        paths[
            "predictions_csv"
        ],
        index=False,
    )


    reasoning_columns = [
        "case_id",
        "source_group_id",
        "case_family",
        "case_variant",
        "gold_label",
        "prediction",
        "participation_assessment",
        "local_temporal_assessment",
        "global_temporal_assessment",
        "temporal_assessment",
        "semantic_assessment",
        "decisive_dimension",
        "schema_exact",
        "correct",
    ]


    results_df[
        reasoning_columns
    ].to_csv(
        paths[
            "reasoning_assessments_csv"
        ],
        index=False,
    )


    confusion_df.to_csv(
        paths[
            "confusion_matrix_csv"
        ]
    )


    family_metrics_df.to_csv(
        paths[
            "family_metrics_csv"
        ],
        index=False,
    )


    variant_metrics_df.to_csv(
        paths[
            "variant_metrics_csv"
        ],
        index=False,
    )


    error_df.to_csv(
        paths[
            "errors_csv"
        ],
        index=False,
    )


    reasoning_inconsistency_df.to_csv(
        paths[
            "reasoning_inconsistencies_csv"
        ],
        index=False,
    )


    assessment_distributions_df.to_csv(
        paths[
            "assessment_distributions_csv"
        ],
        index=False,
    )


    paths[
        "metrics_json"
    ].write_text(
        json.dumps(
            metrics,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


    print("=" * 88)
    print(
        f"{config['experiment_title']} — RESULTS"
    )
    print("=" * 88)

    print(
        "Total cases:",
        metrics[
            "total_cases"
        ],
    )

    print(
        "Valid predictions:",
        metrics[
            "valid_predictions"
        ],
    )

    print(
        "Invalid predictions:",
        metrics[
            "invalid_predictions"
        ],
    )

    print(
        "Exact structured-schema rate:",
        (
            f'{metrics["exact_json_schema_rate"]:.4f}'
        ),
    )

    print(
        "Accuracy:",
        (
            f'{metrics["accuracy_valid_predictions"]:.4f}'
        ),
    )

    print(
        "Balanced accuracy:",
        (
            f'{metrics["balanced_accuracy"]:.4f}'
        ),
    )

    print(
        "ANOMALOUS precision:",
        (
            f'{metrics["anomalous_precision"]:.4f}'
        ),
    )

    print(
        "ANOMALOUS recall:",
        (
            f'{metrics["anomalous_recall"]:.4f}'
        ),
    )

    print(
        "ANOMALOUS F1:",
        (
            f'{metrics["anomalous_f1"]:.4f}'
        ),
    )

    print(
        "NORMAL recall / specificity:",
        (
            f'{metrics["normal_recall_specificity"]:.4f}'
        ),
    )

    print(
        "MCC:",
        (
            f'{metrics["matthews_correlation_coefficient"]:.4f}'
        ),
    )

    print(
        "Matched source-group exact rate:",
        (
            f'{metrics["source_group_exact_match_rate"]:.4f}'
        ),
    )

    print(
        "Reasoning inconsistencies:",
        metrics[
            "reasoning_inconsistency_count"
        ],
    )


    print("\nCONFUSION MATRIX")

    display(
        confusion_df
    )


    print("\nCLASSIFICATION REPORT")

    display(
        classification_report_df
    )


    print("\nPER CASE FAMILY")

    display(
        family_metrics_df
    )


    print("\nPER EXACT CASE VARIANT")

    display(
        variant_metrics_df
    )


    print("\nASSESSMENT DISTRIBUTIONS")

    display(
        assessment_distributions_df
    )


    print(
        "\nSaved cache:",
        paths[
            "prediction_cache"
        ],
    )

    print(
        "Saved prompt:",
        paths[
            "prompt_template"
        ],
    )

    print(
        "Saved prompt diff:",
        paths[
            "prompt_diff"
        ],
    )

    print(
        "Saved predictions:",
        paths[
            "predictions_csv"
        ],
    )

    print(
        "Saved reasoning assessments:",
        paths[
            "reasoning_assessments_csv"
        ],
    )

    print(
        "Saved errors:",
        paths[
            "errors_csv"
        ],
    )


    if len(
        invalid_df
    ):

        print(
            "\nWARNING: Invalid predictions were excluded "
            "from the confusion matrix."
        )


        display(
            invalid_df[
                [
                    "case_id",
                    "case_family",
                    "case_variant",
                    "parse_mode",
                    "raw_output",
                    "generation_error",
                ]
            ]
        )


    else:

        print(
            "\nAll 400 cases produced valid "
            "binary predictions."
        )


    return {
        "metrics": metrics,
        "results_df": results_df,
        "confusion_df": confusion_df,
        "family_metrics_df": family_metrics_df,
        "variant_metrics_df": variant_metrics_df,
        "error_df": error_df,
        "reasoning_inconsistency_df": (
            reasoning_inconsistency_df
        ),
        "assessment_distributions_df": (
            assessment_distributions_df
        ),
    }


# Manual semantic-branch ablation helpers

The helper cell below performs **no automatic prompt rewriting**.

It loads the exact validated L1 No Overlap Structured R1 prompt only as the fixed full baseline to print before the experiment. The final S0 prompt must then be pasted manually in full.

The model-facing S0 payload is mechanically validated to contain:

- `participant_A: {"speaks": ...}`
- `participant_B: {"speaks": ...}`
- the complete local offset-distribution group
- all five global temporal features
- no filtered turns
- no overlap fields
- no `semantic_summaries`


In [ ]:

# ============================================================
# MANUAL SEMANTIC-BRANCH ABLATION HELPERS
#
# IMPORTANT DESIGN:
#   - NO automatic prompt rewriting is performed.
#   - The validated L1 No Overlap prompt is loaded only as the
#     complete fixed Structured R1 baseline to print and inspect.
#   - The final No Semantic Branch prompt must be supplied manually
#     as one complete prompt template.
#   - Every model-facing payload permanently uses:
#       * participant `speaks` only,
#       * the complete local offset-distribution group,
#       * all global temporal features,
#       * no filtered turns,
#       * no overlap evidence,
#       * no coarse or focused semantic summaries.
# ============================================================

from IPython.display import display

SEMANTIC_BASELINE_EXPERIMENT_NAME = (
    "manual_ablation_l1_no_overlap_evidence_speaks_only"
)

SEMANTIC_BASELINE_PROMPT_PATH = (
    OUT_DIR
    / SEMANTIC_BASELINE_EXPERIMENT_NAME
    / "prompt_template.txt"
)

LOCAL_OVERLAP_FIELDS = [
    "clean_overlap_seconds",
    "clean_overlap_percent",
]

LOCAL_OFFSET_DISTRIBUTION_FIELDS = [
    "signed_strict_offsets_seconds",
    "num_signed_strict_offsets",
    "offset_mean_seconds",
    "offset_median_seconds",
    "offset_max_seconds",
    "offset_p75_seconds",
    "offset_p90_seconds",
    "num_offsets_above_1_5_seconds",
    "percent_offsets_above_1_5_seconds",
]

assert (
    LOCAL_OVERLAP_FIELDS
    + LOCAL_OFFSET_DISTRIBUTION_FIELDS
    == LOCAL_FEATURE_FIELDS
)

GLOBAL_MODEL_FEATURE_FIELDS = [
    "best_B_correction_shift_seconds",
    "estimated_B_lateness_seconds",
    "alignment_score_gain_vs_zero",
    "best_num_bilateral_events",
    "best_event_coverage_percent",
]

STANDARD_REASONING_FIELDS_NO_SEMANTIC = [
    "participation_assessment",
    "local_temporal_assessment",
    "global_temporal_assessment",
    "temporal_assessment",
    "decisive_dimension",
]

INSPECTED_MANUAL_SEMANTIC_ABLATIONS = set()


# ============================================================
# NO-SEMANTIC OUTPUT SCHEMA
# ============================================================

NO_SEMANTIC_SCHEMA_KEYS = [
    key
    for key in REASONING_SCHEMA_KEYS
    if key != "semantic_assessment"
]

NO_SEMANTIC_ALLOWED_VALUES = copy.deepcopy(
    REASONING_ALLOWED_VALUES
)

NO_SEMANTIC_ALLOWED_VALUES.pop(
    "semantic_assessment"
)

NO_SEMANTIC_ALLOWED_VALUES[
    "decisive_dimension"
].discard(
    "SEMANTIC"
)

NO_SEMANTIC_REASONING_OUTPUT_BLOCK = (
    STRUCTURED_REASONING_OUTPUT_BLOCK
    .replace(
        """- semantic_assessment:
  - COMPATIBLE
  - INCOMPATIBLE
  - LIMITED

""",
        "",
    )
    .replace(
        """  - SEMANTIC
""",
        "",
    )
    .replace(
        (
            '  "semantic_assessment": '
            '"COMPATIBLE or INCOMPATIBLE or LIMITED",\n'
        ),
        "",
    )
    .replace(
        (
            '"decisive_dimension": '
            '"PARTICIPATION or TEMPORAL or SEMANTIC or NONE"'
        ),
        (
            '"decisive_dimension": '
            '"PARTICIPATION or TEMPORAL or NONE"'
        ),
    )
)

assert (
    "semantic_assessment"
    not in NO_SEMANTIC_REASONING_OUTPUT_BLOCK
)
assert (
    "SEMANTIC"
    not in NO_SEMANTIC_REASONING_OUTPUT_BLOCK
)


# ============================================================
# LOAD THE VALIDATED FULL STRUCTURED R1 BASELINE PROMPT
#
# This is the exact L1 No Overlap baseline:
# speaks only + offsets only + all global features + full semantics.
# ============================================================

def load_semantic_ablation_baseline_prompt_template():
    assert SEMANTIC_BASELINE_PROMPT_PATH.exists(), (
        "The validated L1 No Overlap prompt template was not found:\n"
        f"{SEMANTIC_BASELINE_PROMPT_PATH}\n\n"
        "Run the L1 configuration/inspection cell in the local-temporal "
        "ablation notebook first so its prompt_template.txt is saved."
    )

    template = SEMANTIC_BASELINE_PROMPT_PATH.read_text(
        encoding="utf-8"
    )

    required_markers = [
        "Whether each participant speaks",
        "PARTICIPATION VALIDITY",
        "LOCAL TEMPORAL FEATURES",
        "GLOBAL ALIGNMENT-SHIFT FEATURES",
        "SEMANTIC EVIDENCE",
        "STRUCTURED REASONING OUTPUT",
        '"participation_assessment"',
        '"local_temporal_assessment"',
        '"global_temporal_assessment"',
        '"semantic_assessment"',
        "{participant_A_speaks}",
        "{participant_B_speaks}",
        "{local_temporal_features}",
        "{global_shift_features}",
        "{semantic_summaries}",
    ]

    for marker in required_markers:
        assert marker in template, (
            "The saved Structured R1 baseline prompt is missing "
            f"the marker: {marker}"
        )

    forbidden_markers = [
        "Filtered turns:",
        "{participant_A_turns}",
        "{participant_B_turns}",
        "Use the actual filtered turn lists",
        "Use the speaks fields together with the actual filtered turn lists",
        "TURN FORMAT AND BACKCHANNEL FILTERING",
        "clean_overlap_seconds",
        "clean_overlap_percent",
        "FILTERED OVERLAP",
    ]

    template_lower = template.lower()

    for marker in forbidden_markers:
        assert marker.lower() not in template_lower, (
            "Filtered-turn or overlap content unexpectedly exists in the "
            f"saved semantic-ablation baseline prompt: {marker}"
        )

    return template


# ============================================================
# FIXED FULL BASELINE MODEL INPUT
#
# A deep copy is made. The model-facing baseline permanently removes:
#   - participant filtered_turns,
#   - clean_overlap_seconds,
#   - clean_overlap_percent.
# It retains participant speaks, local offsets, all global features,
# and the original coarse + focused semantic summaries.
# ============================================================

def build_semantic_ablation_baseline_model_input(case):
    payload = copy.deepcopy(
        build_binary_model_input(case)
    )

    for role in [
        "participant_A",
        "participant_B",
    ]:
        participant_payload = payload[role]

        assert "speaks" in participant_payload
        assert "filtered_turns" in participant_payload

        participant_payload.pop(
            "filtered_turns"
        )

        assert set(participant_payload) == {
            "speaks"
        }

    local_features = payload[
        "local_temporal_features"
    ]

    for field in LOCAL_OVERLAP_FIELDS:
        local_features.pop(field)

    assert set(local_features) == set(
        LOCAL_OFFSET_DISTRIBUTION_FIELDS
    )

    assert set(
        payload[
            "global_shift_features"
        ]
    ) == set(
        GLOBAL_MODEL_FEATURE_FIELDS
    )

    assert "semantic_summaries" in payload

    return payload


# ============================================================
# NO SEMANTIC BRANCH MODEL-FACING INPUT
# ============================================================

def build_manual_no_semantic_model_input(case):
    payload = (
        build_semantic_ablation_baseline_model_input(
            case
        )
    )

    payload.pop(
        "semantic_summaries"
    )

    assert "semantic_summaries" not in payload

    return payload


# ============================================================
# BASELINE RENDERER — USED ONLY TO PRINT THE ORIGINAL PROMPT
# ============================================================

def render_semantic_ablation_baseline_prompt_template(
    prompt_template,
    payload,
):
    prompt = prompt_template.format(
        normal_base_reference_text=(
            normal_base_reference_text
        ),
        normal_global_reference_text=(
            normal_global_reference_text
        ),
        duration_seconds=(
            payload[
                "analysis_duration_seconds"
            ]
        ),
        participant_A_speaks=canonical_json(
            payload[
                "participant_A"
            ][
                "speaks"
            ]
        ),
        participant_B_speaks=canonical_json(
            payload[
                "participant_B"
            ][
                "speaks"
            ]
        ),
        local_temporal_features=json.dumps(
            payload[
                "local_temporal_features"
            ],
            indent=2,
            ensure_ascii=False,
        ),
        global_shift_features=json.dumps(
            payload[
                "global_shift_features"
            ],
            indent=2,
            ensure_ascii=False,
        ),
        semantic_summaries=json.dumps(
            payload[
                "semantic_summaries"
            ],
            indent=2,
            ensure_ascii=False,
        ),
    )

    prompt_lower = prompt.lower()

    for forbidden_key in FORBIDDEN_PROMPT_KEYS:
        assert forbidden_key.lower() not in prompt_lower

    for forbidden_marker in [
        "filtered turns:",
        "clean_overlap_seconds",
        "clean_overlap_percent",
        "filtered overlap",
    ]:
        assert forbidden_marker not in prompt_lower

    return prompt


# ============================================================
# NO-SEMANTIC MANUAL TEMPLATE RENDERER
# ============================================================

def render_manual_no_semantic_prompt_template(
    prompt_template,
    payload,
):
    assert isinstance(prompt_template, str)
    assert prompt_template.strip()

    prompt = prompt_template.format(
        normal_base_reference_text=(
            normal_base_reference_text
        ),
        normal_global_reference_text=(
            normal_global_reference_text
        ),
        duration_seconds=(
            payload[
                "analysis_duration_seconds"
            ]
        ),
        participant_A_speaks=canonical_json(
            payload[
                "participant_A"
            ][
                "speaks"
            ]
        ),
        participant_B_speaks=canonical_json(
            payload[
                "participant_B"
            ][
                "speaks"
            ]
        ),
        local_temporal_features=json.dumps(
            payload[
                "local_temporal_features"
            ],
            indent=2,
            ensure_ascii=False,
        ),
        global_shift_features=json.dumps(
            payload[
                "global_shift_features"
            ],
            indent=2,
            ensure_ascii=False,
        ),
    )

    prompt_lower = prompt.lower()

    for forbidden_key in FORBIDDEN_PROMPT_KEYS:
        assert forbidden_key.lower() not in prompt_lower, (
            "Forbidden field name leaked into prompt: "
            f"{forbidden_key}"
        )

    for forbidden_marker in [
        "filtered turns:",
        "participant_a_filtered_turns",
        "participant_b_filtered_turns",
        "{participant_a_turns}",
        "{participant_b_turns}",
        "clean_overlap_seconds",
        "clean_overlap_percent",
        "filtered overlap",
    ]:
        assert forbidden_marker not in prompt_lower, (
            "Forbidden turn/overlap evidence leaked into prompt: "
            f"{forbidden_marker}"
        )

    assert "semantic" not in prompt_lower, (
        "Semantic content or semantic instructions remain in the "
        "No Semantic Branch model prompt."
    )

    return prompt


# ============================================================
# PRINT THE ORIGINAL FIXED STRUCTURED R1 BASELINE PROMPT
# ============================================================

def print_original_semantic_ablation_baseline_prompt(
    *,
    experiment_title,
    case_index=0,
):
    ordered_cases = sorted(
        consolidation_cases,
        key=lambda case: str(
            case[
                "case_id"
            ]
        ),
    )

    assert 0 <= case_index < len(ordered_cases)

    case = ordered_cases[case_index]

    baseline_template = (
        load_semantic_ablation_baseline_prompt_template()
    )

    baseline_payload = (
        build_semantic_ablation_baseline_model_input(
            case
        )
    )

    baseline_prompt = (
        render_semantic_ablation_baseline_prompt_template(
            baseline_template,
            baseline_payload,
        )
    )

    export_dir = (
        OUT_DIR
        / "manual_semantic_ablation_baseline_prompts"
    )
    export_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    safe_name = re.sub(
        r"[^a-z0-9]+",
        "_",
        experiment_title.lower(),
    ).strip("_")

    export_path = (
        export_dir
        / f"{safe_name}_original_semantic_ablation_baseline_prompt.txt"
    )

    export_path.write_text(
        baseline_prompt,
        encoding="utf-8",
    )

    print("=" * 100)
    print(
        "ORIGINAL STRUCTURED R1 SEMANTIC-ABLATION BASELINE PROMPT"
    )
    print(
        "Target experiment:",
        experiment_title,
    )
    print("=" * 100)
    print(
        "Inspection case ID:",
        case[
            "case_id"
        ],
    )
    print(
        "Participant evidence: speaks only."
    )
    print(
        "Local evidence: complete offset distribution only."
    )
    print(
        "Global evidence: all five global temporal features."
    )
    print(
        "Semantic evidence in original baseline: coarse + focused."
    )
    print(
        "Filtered turns supplied: False"
    )
    print(
        "Overlap evidence supplied: False"
    )
    print(
        "This is the fixed complete baseline prompt that must be "
        "edited manually to remove the complete semantic branch."
    )

    print("\nEXACT BASELINE MODEL-FACING PAYLOAD")
    print(
        json.dumps(
            baseline_payload,
            indent=2,
            ensure_ascii=False,
        )
    )

    print("\n" + "=" * 100)
    print("EXACT ORIGINAL RENDERED PROMPT")
    print("=" * 100)
    print(baseline_prompt)

    print("\n" + "=" * 100)
    print("BASELINE HASHES")
    print("=" * 100)
    print(
        "Baseline template SHA256:",
        sha256_text(
            baseline_template
        ),
    )
    print(
        "Rendered baseline prompt SHA256:",
        sha256_text(
            baseline_prompt
        ),
    )
    print(
        "Baseline payload SHA256:",
        sha256_text(
            canonical_json(
                baseline_payload
            )
        ),
    )
    print(
        "Saved rendered prompt:",
        export_path,
    )

    return {
        "case_id": str(
            case[
                "case_id"
            ]
        ),
        "prompt_template": baseline_template,
        "prompt": baseline_prompt,
        "payload": baseline_payload,
        "export_path": export_path,
    }


# ============================================================
# MANUAL NO-SEMANTIC PROMPT VALIDATION
#
# The function validates the final manually written prompt.
# It does not edit, rewrite, or delete any prompt text.
# ============================================================

def validate_manual_no_semantic_prompt_template(
    prompt_template,
):
    if prompt_template is None:
        raise RuntimeError(
            "The manual prompt template is still None. First run the "
            "ORIGINAL PROMPT cell, inspect/send its output for manual "
            "editing, and then paste the complete No Semantic Branch "
            "template into the manual-prompt cell."
        )

    assert isinstance(prompt_template, str)

    template = prompt_template.strip()

    if not template:
        raise RuntimeError(
            "The manual prompt template is empty."
        )

    placeholder_text = template.upper()

    if (
        "PASTE THE MANUALLY" in placeholder_text
        or "PASTE MANUAL" in placeholder_text
        or "TODO_MANUAL_PROMPT" in placeholder_text
    ):
        raise RuntimeError(
            "The manual prompt placeholder has not been replaced."
        )

    required_placeholders = [
        "{duration_seconds}",
        "{participant_A_speaks}",
        "{participant_B_speaks}",
        "{local_temporal_features}",
        "{global_shift_features}",
    ]

    for placeholder in required_placeholders:
        assert placeholder in template, (
            "Required prompt placeholder missing: "
            f"{placeholder}"
        )

    assert "{semantic_summaries}" not in template

    forbidden_turn_fragments = [
        "Filtered turns:",
        "{participant_A_turns}",
        "{participant_B_turns}",
        "Use the actual filtered turn lists",
        "Use the speaks fields together with the actual filtered turn lists",
        "TURN FORMAT AND BACKCHANNEL FILTERING",
    ]

    for fragment in forbidden_turn_fragments:
        assert fragment not in template, (
            "Filtered-turn instruction or placeholder exists in the "
            f"manual prompt: {fragment}"
        )

    template_lower = template.lower()

    for fragment in [
        "clean_overlap_seconds",
        "clean_overlap_percent",
        "filtered overlap",
    ]:
        assert fragment not in template_lower, (
            "Overlap instruction or field exists in the manual "
            f"No Semantic Branch prompt: {fragment}"
        )

    assert "semantic" not in template_lower, (
        "The manually defined No Semantic Branch prompt still contains "
        "semantic evidence, semantic instructions, or semantic output."
    )

    required_markers = [
        "PARTICIPATION VALIDITY",
        "LOCAL TEMPORAL FEATURES",
        "GLOBAL ALIGNMENT-SHIFT FEATURES",
        "FROZEN NORMAL GLOBAL-ALIGNMENT REFERENCE",
        "JOINT TEMPORAL COORDINATION DECISION",
        "STRUCTURED REASONING OUTPUT",
        '"participation_assessment"',
        '"local_temporal_assessment"',
        '"global_temporal_assessment"',
        '"temporal_assessment"',
        '"decisive_dimension"',
        '"label"',
    ]

    for marker in required_markers:
        assert marker in template, (
            "Required Structured R1 marker missing from the manual "
            f"No Semantic Branch prompt: {marker}"
        )

    example_payload = (
        build_manual_no_semantic_model_input(
            consolidation_cases[0]
        )
    )

    example_prompt = (
        render_manual_no_semantic_prompt_template(
            template,
            example_payload,
        )
    )

    assert "semantic_summaries" not in example_payload
    assert "semantic" not in example_prompt.lower()

    return {
        "template": template,
        "example_prompt": example_prompt,
        "example_payload": example_payload,
    }


# ============================================================
# BUILD ONE MANUAL NO-SEMANTIC PROMPT
# ============================================================

def build_manual_no_semantic_prompt(
    case,
    config,
):
    payload = (
        build_manual_no_semantic_model_input(
            case
        )
    )

    prompt = (
        render_manual_no_semantic_prompt_template(
            config[
                "reasoning_prompt_template"
            ],
            payload,
        )
    )

    return prompt, payload


# ============================================================
# PREPARE THE MANUAL NO-SEMANTIC EXPERIMENT
# ============================================================

def prepare_manual_no_semantic_experiment(
    *,
    experiment_version,
    experiment_title,
    ablation_id,
    manual_prompt_template,
):
    validated = (
        validate_manual_no_semantic_prompt_template(
            manual_prompt_template
        )
    )

    ablation_prompt_template = validated[
        "template"
    ]

    baseline_prompt_template = (
        load_semantic_ablation_baseline_prompt_template()
    )

    schema_keys = list(
        NO_SEMANTIC_SCHEMA_KEYS
    )

    allowed_values = copy.deepcopy(
        NO_SEMANTIC_ALLOWED_VALUES
    )

    experiment_dir = (
        OUT_DIR
        / experiment_version
    )

    experiment_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    paths = {
        "prediction_cache": (
            experiment_dir
            / "predictions_cache.json"
        ),
        "predictions_csv": (
            experiment_dir
            / "predictions_all_400.csv"
        ),
        "metrics_json": (
            experiment_dir
            / "metrics.json"
        ),
        "confusion_matrix_csv": (
            experiment_dir
            / "confusion_matrix.csv"
        ),
        "family_metrics_csv": (
            experiment_dir
            / "metrics_by_case_family.csv"
        ),
        "variant_metrics_csv": (
            experiment_dir
            / "metrics_by_case_variant.csv"
        ),
        "errors_csv": (
            experiment_dir
            / "classification_errors.csv"
        ),
        "reasoning_assessments_csv": (
            experiment_dir
            / "reasoning_assessments.csv"
        ),
        "reasoning_inconsistencies_csv": (
            experiment_dir
            / "reasoning_inconsistencies.csv"
        ),
        "assessment_distributions_csv": (
            experiment_dir
            / "assessment_distributions.csv"
        ),
        "prompt_template": (
            experiment_dir
            / "prompt_template.txt"
        ),
        "baseline_prompt_copy": (
            experiment_dir
            / "original_full_r1_baseline_prompt_template.txt"
        ),
        "prompt_diff": (
            experiment_dir
            / "manual_prompt_diff_vs_full_r1_baseline.txt"
        ),
        "experiment_manifest": (
            experiment_dir
            / "experiment_manifest.json"
        ),
    }

    paths[
        "prompt_template"
    ].write_text(
        ablation_prompt_template,
        encoding="utf-8",
    )

    paths[
        "baseline_prompt_copy"
    ].write_text(
        baseline_prompt_template,
        encoding="utf-8",
    )

    prompt_diff_lines = difflib.unified_diff(
        baseline_prompt_template.splitlines(),
        ablation_prompt_template.splitlines(),
        fromfile=(
            "original_full_r1_baseline_prompt_template.txt"
        ),
        tofile=(
            "manually_defined_no_semantic_branch_prompt_template.txt"
        ),
        lineterm="",
    )

    paths[
        "prompt_diff"
    ].write_text(
        "\n".join(prompt_diff_lines),
        encoding="utf-8",
    )

    baseline_prompt_sha256 = sha256_text(
        baseline_prompt_template
    )

    reasoning_prompt_sha256 = sha256_text(
        ablation_prompt_template
    )

    reasoning_schema_sha256 = sha256_text(
        NO_SEMANTIC_REASONING_OUTPUT_BLOCK
    )

    normal_reference_sha256 = sha256_text(
        normal_base_reference_text
        + "\n"
        + normal_global_reference_text
    )

    manifest = {
        "experiment_version": experiment_version,
        "experiment_title": experiment_title,
        "ablation_id": ablation_id,
        "semantic_ablation_mode": (
            "NO_SEMANTIC_BRANCH"
        ),
        "source_experiment": (
            SEMANTIC_BASELINE_EXPERIMENT_NAME
        ),
        "source_prompt_sha256": (
            baseline_prompt_sha256
        ),
        "full_r1_prompt_sha256": (
            baseline_prompt_sha256
        ),
        "reasoning_prompt_sha256": (
            reasoning_prompt_sha256
        ),
        "reasoning_schema_sha256": (
            reasoning_schema_sha256
        ),
        "normal_reference_sha256": (
            normal_reference_sha256
        ),
        "semantic_input": "none",
        "focused_summaries_used": False,
        "semantic_branch_removed": True,
        "temporal_profiles_used": [
            "NORMAL"
        ],
        "assessment_policy": (
            "Structured R1 No Semantic Branch policy using participant "
            "speaks only, the complete local offset-distribution group, "
            "all global temporal features, no overlap evidence, and no "
            "coarse or focused semantic summaries."
        ),
        "schema_keys": schema_keys,
        "allowed_values": {
            key: sorted(list(values))
            for key, values
            in allowed_values.items()
        },
        "max_new_tokens": (
            MAX_NEW_TOKENS_REASONING
        ),
        "model_id": MODEL_ID,
        "decoding": {
            "do_sample": False,
            "use_cache": True,
        },
        "participant_model_input": (
            "speaks_only"
        ),
        "local_model_input": (
            "offset_distribution_only"
        ),
        "global_model_input": (
            "all_global_temporal_features"
        ),
        "filtered_turns_excluded_from_all_model_facing_inputs": True,
        "overlap_excluded_from_all_model_facing_inputs": True,
        "prompt_definition_method": (
            "manual_complete_template"
        ),
        "automatic_prompt_rewriting_used": False,
    }

    paths[
        "experiment_manifest"
    ].write_text(
        json.dumps(
            manifest,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    config = {
        **manifest,
        "experiment_dir": experiment_dir,
        "baseline_prompt_template": (
            baseline_prompt_template
        ),
        "reasoning_prompt_template": (
            ablation_prompt_template
        ),
        "paths": paths,
    }

    example_prompt, example_payload = (
        build_manual_no_semantic_prompt(
            consolidation_cases[0],
            config,
        )
    )

    assert set(
        example_payload[
            "participant_A"
        ]
    ) == {"speaks"}

    assert set(
        example_payload[
            "participant_B"
        ]
    ) == {"speaks"}

    assert set(
        example_payload[
            "local_temporal_features"
        ]
    ) == set(
        LOCAL_OFFSET_DISTRIBUTION_FIELDS
    )

    assert set(
        example_payload[
            "global_shift_features"
        ]
    ) == set(
        GLOBAL_MODEL_FEATURE_FIELDS
    )

    assert "semantic_summaries" not in example_payload
    assert "semantic" not in example_prompt.lower()

    print("=" * 88)
    print(
        f"{experiment_title} — MANUAL CONFIGURATION READY"
    )
    print("=" * 88)
    print(
        "Ablation ID:",
        ablation_id,
    )
    print(
        "Semantic ablation mode:",
        "NO_SEMANTIC_BRANCH",
    )
    print(
        "Participant evidence:",
        "speaks only",
    )
    print(
        "Local evidence:",
        "complete offset distribution only",
    )
    print(
        "Global evidence:",
        "all global temporal features",
    )
    print(
        "Semantic evidence passed to model:",
        False,
    )
    print(
        "Filtered turns passed to model:",
        False,
    )
    print(
        "Overlap passed to model:",
        False,
    )
    print(
        "Automatic prompt rewriting used:",
        False,
    )
    print(
        "Schema keys:",
        schema_keys,
    )
    print(
        "Baseline prompt SHA256:",
        baseline_prompt_sha256,
    )
    print(
        "Manual prompt SHA256:",
        reasoning_prompt_sha256,
    )
    print(
        "Prompt diff:",
        paths[
            "prompt_diff"
        ],
    )

    return config


# ============================================================
# REQUIRED MANUAL PROMPT INSPECTION
# ============================================================

def inspect_manual_no_semantic_prompt(
    config,
    *,
    case_index=0,
):
    ordered_cases = sorted(
        consolidation_cases,
        key=lambda case: str(
            case[
                "case_id"
            ]
        ),
    )

    assert 0 <= case_index < len(ordered_cases)

    case = ordered_cases[case_index]

    prompt, payload = (
        build_manual_no_semantic_prompt(
            case,
            config,
        )
    )

    print("=" * 100)
    print(
        "MANUAL PROMPT INSPECTION —",
        config[
            "experiment_title"
        ],
    )
    print("=" * 100)
    print(
        "Inspection case ID:",
        case[
            "case_id"
        ],
    )
    print(
        "Gold label is intentionally NOT printed inside the model prompt."
    )
    print(
        "Participant evidence supplied: speaks only."
    )
    print(
        "Local evidence supplied: complete offset distribution only."
    )
    print(
        "Global evidence supplied: all global temporal features."
    )
    print(
        "Semantic evidence supplied: False"
    )
    print(
        "Filtered turns supplied: False"
    )
    print(
        "Overlap evidence supplied: False"
    )
    print(
        "Prompt definition method: manual complete template"
    )

    print("\nEXACT ABLATED MODEL INPUT PAYLOAD")
    print(
        json.dumps(
            payload,
            indent=2,
            ensure_ascii=False,
        )
    )

    print("\n" + "=" * 100)
    print("EXACT MANUALLY DEFINED RENDERED MODEL PROMPT")
    print("=" * 100)
    print(prompt)

    print("\n" + "=" * 100)
    print("PROMPT INSPECTION HASHES")
    print("=" * 100)
    print(
        "Rendered prompt SHA256:",
        sha256_text(prompt),
    )
    print(
        "Ablated payload SHA256:",
        sha256_text(
            canonical_json(payload)
        ),
    )
    print(
        "Manual template SHA256:",
        config[
            "reasoning_prompt_sha256"
        ],
    )
    print(
        "Template diff file:",
        config[
            "paths"
        ][
            "prompt_diff"
        ],
    )

    INSPECTED_MANUAL_SEMANTIC_ABLATIONS.add(
        config[
            "experiment_version"
        ]
    )

    return {
        "case_id": str(
            case[
                "case_id"
            ]
        ),
        "prompt": prompt,
        "payload": payload,
    }


# ============================================================
# CONFIG-AWARE STRUCTURED OUTPUT PARSER
# ============================================================

def parse_manual_no_semantic_prediction(
    raw_output,
    config,
):
    parsed = extract_first_json_object(
        raw_output
    )

    schema_keys = config[
        "schema_keys"
    ]

    allowed_values = {
        key: set(values)
        for key, values
        in config[
            "allowed_values"
        ].items()
    }

    normalized = {
        key: None
        for key in schema_keys
    }

    schema_errors = []

    if isinstance(parsed, dict):
        for key in schema_keys:
            if key in parsed:
                normalized[key] = str(
                    parsed[key]
                ).strip().upper()

        for key in schema_keys:
            if normalized[key] not in allowed_values[key]:
                schema_errors.append(
                    f"{key}: {normalized[key]}"
                )

        exact_keys = (
            set(parsed.keys())
            == set(schema_keys)
        )

        schema_exact = (
            exact_keys
            and not schema_errors
        )

        label = normalized[
            "label"
        ]

        if label in LABELS:
            result = {
                "prediction": label,
                "parse_mode": (
                    "structured_json"
                ),
                "schema_exact": bool(
                    schema_exact
                ),
                "parsed_output": parsed,
                "schema_errors": (
                    schema_errors
                ),
            }

            for field in (
                STANDARD_REASONING_FIELDS_NO_SEMANTIC
            ):
                result[field] = normalized.get(field)

            result[
                "semantic_assessment"
            ] = None

            return result

    plain_output = (
        raw_output
        .strip()
        .strip('"')
        .strip("'")
        .upper()
    )

    empty_fields = {
        field: None
        for field in (
            STANDARD_REASONING_FIELDS_NO_SEMANTIC
        )
    }

    empty_fields[
        "semantic_assessment"
    ] = None

    if plain_output in LABELS:
        return {
            "prediction": plain_output,
            "parse_mode": (
                "exact_plaintext_fallback"
            ),
            "schema_exact": False,
            "parsed_output": None,
            "schema_errors": [
                "Structured reasoning fields missing."
            ],
            **empty_fields,
        }

    return {
        "prediction": None,
        "parse_mode": "invalid",
        "schema_exact": False,
        "parsed_output": parsed,
        "schema_errors": (
            schema_errors
            if schema_errors
            else [
                "No valid structured JSON prediction."
            ]
        ),
        **{
            field: normalized.get(field)
            for field in (
                STANDARD_REASONING_FIELDS_NO_SEMANTIC
            )
        },
        "semantic_assessment": None,
    }


# ============================================================
# CHECKPOINT CACHE
# ============================================================

def create_manual_no_semantic_cache(
    config,
):
    return {
        "experiment_version": (
            config[
                "experiment_version"
            ]
        ),
        "experiment_title": (
            config[
                "experiment_title"
            ]
        ),
        "ablation_id": (
            config[
                "ablation_id"
            ]
        ),
        "semantic_ablation_mode": (
            "NO_SEMANTIC_BRANCH"
        ),
        "source_experiment": (
            config[
                "source_experiment"
            ]
        ),
        "model_id": MODEL_ID,
        "source_prompt_sha256": (
            config[
                "source_prompt_sha256"
            ]
        ),
        "full_r1_prompt_sha256": (
            config[
                "full_r1_prompt_sha256"
            ]
        ),
        "reasoning_prompt_sha256": (
            config[
                "reasoning_prompt_sha256"
            ]
        ),
        "reasoning_schema_sha256": (
            config[
                "reasoning_schema_sha256"
            ]
        ),
        "normal_reference_sha256": (
            config[
                "normal_reference_sha256"
            ]
        ),
        "semantic_input": "none",
        "focused_summaries_used": False,
        "semantic_branch_removed": True,
        "temporal_profiles_used": [
            "NORMAL"
        ],
        "assessment_policy": (
            config[
                "assessment_policy"
            ]
        ),
        "schema_keys": (
            config[
                "schema_keys"
            ]
        ),
        "participant_model_input": (
            "speaks_only"
        ),
        "local_model_input": (
            "offset_distribution_only"
        ),
        "global_model_input": (
            "all_global_temporal_features"
        ),
        "filtered_turns_excluded": True,
        "overlap_excluded": True,
        "prompt_definition_method": (
            "manual_complete_template"
        ),
        "max_new_tokens": (
            MAX_NEW_TOKENS_REASONING
        ),
        "created_at_utc": (
            reasoning_utc_now()
        ),
        "updated_at_utc": (
            reasoning_utc_now()
        ),
        "records": {},
    }


# ============================================================
# INFERENCE — SAME QWEN CALL / ORDER / DECODING / CHECKPOINTING
# ============================================================

def run_manual_no_semantic_experiment(
    config,
    *,
    print_each_case_prompt=False,
):
    assert (
        config[
            "experiment_version"
        ]
        in INSPECTED_MANUAL_SEMANTIC_ABLATIONS
    ), (
        "Manual prompt inspection has not been completed in this "
        "runtime. Run the inspection cell immediately above before "
        "starting inference."
    )

    prediction_cache_path = (
        config[
            "paths"
        ][
            "prediction_cache"
        ]
    )

    expected_cache_header = (
        create_manual_no_semantic_cache(
            config
        )
    )

    if prediction_cache_path.exists():
        prediction_cache = json.loads(
            prediction_cache_path.read_text(
                encoding="utf-8"
            )
        )

        for key in [
            "experiment_version",
            "ablation_id",
            "semantic_ablation_mode",
            "source_experiment",
            "model_id",
            "source_prompt_sha256",
            "full_r1_prompt_sha256",
            "reasoning_prompt_sha256",
            "reasoning_schema_sha256",
            "normal_reference_sha256",
            "semantic_input",
            "focused_summaries_used",
            "semantic_branch_removed",
            "temporal_profiles_used",
            "assessment_policy",
            "schema_keys",
            "participant_model_input",
            "local_model_input",
            "global_model_input",
            "filtered_turns_excluded",
            "overlap_excluded",
            "prompt_definition_method",
            "max_new_tokens",
        ]:
            assert (
                prediction_cache[key]
                == expected_cache_header[key]
            ), (
                f"Cache mismatch for {key}. Delete the old cache "
                "only if you intentionally changed the manual prompt."
            )

        print(
            "Resuming cache:",
            prediction_cache_path,
        )
        print(
            "Existing records:",
            len(
                prediction_cache[
                    "records"
                ]
            ),
        )

    else:
        prediction_cache = expected_cache_header

        reasoning_atomic_write_json(
            prediction_cache_path,
            prediction_cache,
        )

        print(
            "Created cache:",
            prediction_cache_path,
        )

    ordered_cases = sorted(
        consolidation_cases,
        key=lambda case: str(
            case[
                "case_id"
            ]
        ),
    )

    assert len(ordered_cases) == 400

    for case in tqdm(
        ordered_cases,
        desc=config[
            "experiment_version"
        ],
    ):
        case_id = str(
            case[
                "case_id"
            ]
        )

        prompt, model_input_payload = (
            build_manual_no_semantic_prompt(
                case,
                config,
            )
        )

        baseline_payload = (
            build_semantic_ablation_baseline_model_input(
                case
            )
        )

        assert (
            model_input_payload[
                "participant_A"
            ]
            == baseline_payload[
                "participant_A"
            ]
        )

        assert (
            model_input_payload[
                "participant_B"
            ]
            == baseline_payload[
                "participant_B"
            ]
        )

        assert (
            model_input_payload[
                "local_temporal_features"
            ]
            == baseline_payload[
                "local_temporal_features"
            ]
        )

        assert (
            model_input_payload[
                "global_shift_features"
            ]
            == baseline_payload[
                "global_shift_features"
            ]
        )

        assert "semantic_summaries" not in model_input_payload

        current_payload_keys = (
            collect_nested_keys_reasoning(
                model_input_payload
            )
        )

        for semantic_key in (
            FULL_SEMANTIC_REQUIRED_KEYS
        ):
            assert semantic_key not in current_payload_keys

        assert set(
            model_input_payload[
                "participant_A"
            ]
        ) == {"speaks"}

        assert set(
            model_input_payload[
                "participant_B"
            ]
        ) == {"speaks"}

        assert set(
            model_input_payload[
                "local_temporal_features"
            ]
        ) == set(
            LOCAL_OFFSET_DISTRIBUTION_FIELDS
        )

        assert set(
            model_input_payload[
                "global_shift_features"
            ]
        ) == set(
            GLOBAL_MODEL_FEATURE_FIELDS
        )

        assert "Filtered turns:" not in prompt

        for forbidden_marker in [
            "clean_overlap_seconds",
            "clean_overlap_percent",
            "filtered overlap",
            "semantic",
        ]:
            assert forbidden_marker not in prompt.lower()

        prompt_sha256 = sha256_text(
            prompt
        )

        input_payload_sha256 = sha256_text(
            canonical_json(
                model_input_payload
            )
        )

        existing_record = prediction_cache[
            "records"
        ].get(case_id)

        if (
            existing_record is not None
            and existing_record.get(
                "prediction"
            ) in LABELS
        ):
            assert (
                existing_record[
                    "prompt_sha256"
                ]
                == prompt_sha256
            )

            assert (
                existing_record[
                    "input_payload_sha256"
                ]
                == input_payload_sha256
            )

            continue

        if print_each_case_prompt:
            print("\n" + "=" * 100)
            print("CASE:", case_id)
            print("=" * 100)
            print(prompt)

        started = time.perf_counter()

        try:
            raw_output, input_token_count = (
                qwen_text_only_binary(
                    prompt,
                    max_new_tokens=(
                        MAX_NEW_TOKENS_REASONING
                    ),
                )
            )

            parsed_result = (
                parse_manual_no_semantic_prediction(
                    raw_output,
                    config,
                )
            )

            generation_error = None

        except Exception as exc:
            raw_output = ""
            input_token_count = None

            parsed_result = {
                "prediction": None,
                "parse_mode": (
                    "generation_error"
                ),
                "schema_exact": False,
                "parsed_output": None,
                "schema_errors": [],
                "participation_assessment": None,
                "local_temporal_assessment": None,
                "global_temporal_assessment": None,
                "temporal_assessment": None,
                "semantic_assessment": None,
                "decisive_dimension": None,
            }

            generation_error = (
                f"{type(exc).__name__}: {exc}"
            )

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        elapsed_seconds = (
            time.perf_counter()
            - started
        )

        prediction_cache[
            "records"
        ][case_id] = {
            "case_id": case_id,
            "source_group_id": str(
                case[
                    "source_group_id"
                ]
            ),
            "case_family": get_case_family(case),
            "case_variant": str(
                case[
                    "case_variant"
                ]
            ),
            "gold_binary_label": str(
                case[
                    "gold_binary_label"
                ]
            ).upper(),
            "ablation_id": (
                config[
                    "ablation_id"
                ]
            ),
            "semantic_ablation_mode": (
                "NO_SEMANTIC_BRANCH"
            ),
            "participant_model_input": (
                "speaks_only"
            ),
            "local_model_input": (
                "offset_distribution_only"
            ),
            "global_model_input": (
                "all_global_temporal_features"
            ),
            "semantic_input": "none",
            "focused_summaries_used": False,
            "semantic_branch_removed": True,
            "filtered_turns_excluded": True,
            "overlap_excluded": True,
            "prompt_definition_method": (
                "manual_complete_template"
            ),
            "prompt_sha256": prompt_sha256,
            "input_payload_sha256": (
                input_payload_sha256
            ),
            "input_token_count": (
                input_token_count
            ),
            "max_new_tokens": (
                MAX_NEW_TOKENS_REASONING
            ),
            "raw_output": raw_output,
            "prediction": (
                parsed_result[
                    "prediction"
                ]
            ),
            "participation_assessment": (
                parsed_result[
                    "participation_assessment"
                ]
            ),
            "local_temporal_assessment": (
                parsed_result[
                    "local_temporal_assessment"
                ]
            ),
            "global_temporal_assessment": (
                parsed_result[
                    "global_temporal_assessment"
                ]
            ),
            "temporal_assessment": (
                parsed_result[
                    "temporal_assessment"
                ]
            ),
            "semantic_assessment": None,
            "decisive_dimension": (
                parsed_result[
                    "decisive_dimension"
                ]
            ),
            "parse_mode": (
                parsed_result[
                    "parse_mode"
                ]
            ),
            "schema_exact": bool(
                parsed_result[
                    "schema_exact"
                ]
            ),
            "schema_errors": (
                parsed_result[
                    "schema_errors"
                ]
            ),
            "parsed_output": (
                parsed_result[
                    "parsed_output"
                ]
            ),
            "generation_error": generation_error,
            "elapsed_seconds": round(
                elapsed_seconds,
                4,
            ),
            "completed_at_utc": (
                reasoning_utc_now()
            ),
        }

        prediction_cache[
            "updated_at_utc"
        ] = reasoning_utc_now()

        reasoning_atomic_write_json(
            prediction_cache_path,
            prediction_cache,
        )

    print("\n" + "=" * 88)
    print(
        f"{config['experiment_title']} — INFERENCE COMPLETE"
    )
    print("=" * 88)
    print(
        "Cached records:",
        len(
            prediction_cache[
                "records"
            ]
        ),
    )
    print(
        "Prediction cache:",
        prediction_cache_path,
    )

    return prediction_cache


# Required workflow

Execute the experiment strictly in this order:

1. Run **S0.1** to print the exact original full Structured R1 baseline prompt.
2. Manually define the complete S0 prompt in **S0.2**.
3. Run **S0.3** to validate and save the exact manual template.
4. Run **S0.4** and inspect the exact payload and rendered prompt.
5. Only after inspection, run **S0.5** for the 400-case inference.
6. Run **S0.6** for evaluation.

Inference refuses to start unless prompt inspection has occurred in the current runtime. Every completed case is checkpointed to Google Drive, so interrupted runs resume safely.


# S0 — No Semantic Branch

The complete semantic component is removed.

Removed from the model-facing input:

- all coarse summaries:
  - `speech_content_summary`
  - `apparent_topic`
- all focused summaries:
  - `detailed_speech_summary`
  - `main_topic`
  - `secondary_topics`
  - `key_semantic_details`
  - `summary_specificity`
  - `unclear_content`
  - `confidence`

Removed from the structured output:

- `semantic_assessment`

Removed from `decisive_dimension`:

- `SEMANTIC`

Retained:

- participant `speaks` only
- the complete local offset distribution
- all five global temporal features
- `participation_assessment`
- `local_temporal_assessment`
- `global_temporal_assessment`
- `temporal_assessment`
- `decisive_dimension`
- final binary `label`

This diagnostic asks whether the current combined model is already solving almost everything through participation and temporal evidence.


In [ ]:
# STEP S0.1 — PRINT THE ORIGINAL FIXED FULL STRUCTURED R1 BASELINE PROMPT
# Run this cell first and use its printed prompt for the manual adaptation.

S0_ORIGINAL_PROMPT = print_original_semantic_ablation_baseline_prompt(
    experiment_title=(
        "S0 — No Semantic Branch"
    ),
    case_index=0,
)


ORIGINAL STRUCTURED R1 SEMANTIC-ABLATION BASELINE PROMPT
Target experiment: S0 — No Semantic Branch
Inspection case ID: consolidation_lag_2sec_000
Participant evidence: speaks only.
Local evidence: complete offset distribution only.
Global evidence: all five global temporal features.
Semantic evidence in original baseline: coarse + focused.
Filtered turns supplied: False
Overlap evidence supplied: False
This is the fixed complete baseline prompt that must be edited manually to remove the complete semantic branch.

EXACT BASELINE MODEL-FACING PAYLOAD
{
  "analysis_duration_seconds": 120.0,
  "participant_A": {
    "speaks": true
  },
  "participant_B": {
    "speaks": true
  },
  "local_temporal_features": {
    "signed_strict_offsets_seconds": [
      3.16,
      4.28
    ],
    "num_signed_strict_offsets": 2,
    "offset_mean_seconds": 3.72,
    "offset_median_seconds": 3.72,
    "offset_max_seconds": 4.28,
    "offset_p75_seconds": 4.0,
    "offset_p90_seconds": 4.17,
    "num_offset

In [ ]:
# STEP S0.2 — MANUAL PROMPT TEMPLATE
# Leave this as None until the complete S0 prompt has been manually defined.
#
# The final prompt must remove:
#   - the complete SEMANTIC EVIDENCE section,
#   - SEMANTIC SUMMARIES from CURRENT CASE,
#   - semantic_assessment from the structured output,
#   - SEMANTIC from decisive_dimension,
#   - every semantic-dependent final-decision instruction.
#
# The final prompt must retain:
#   - participation,
#   - local offsets,
#   - all global temporal features,
#   - the original temporal reasoning policy.

S0_MANUAL_PROMPT_TEMPLATE = r"""You are evaluating whether two participant records are jointly
consistent with one coherent, naturally synchronized, spoken
120-second dyadic interaction.

You are given information extracted independently for Participant A
and Participant B from the same 120-second analysis timeline.

Your task is binary classification only:

- NORMAL
- ANOMALOUS

Do not predict, infer, or name a specific anomaly type.

============================================================
AVAILABLE EVIDENCE
============================================================

You receive:

1. Whether each participant speaks during the 120-second interval.
2. Local turn-handoff offset-distribution features.
3. Global temporal alignment-shift features.
4. Frozen temporal reference statistics calculated only from
   separate NORMAL dyadic conversations.

Use only the supplied evidence.

Do not infer anything from identifiers, filenames, paths, dataset order,
sample position, participant identity, or hidden labels.

None of those fields are provided.

============================================================
OPERATIONAL DEFINITION OF NORMAL
============================================================

A NORMAL case must be compatible with one coherent, naturally
synchronized, two-person spoken interaction.

NORMAL requires two independent properties:

1. Participation validity
2. Temporal coordination

Evaluate these two properties separately before making the final
binary decision.

A case should be classified as NORMAL only when both properties are
sufficiently supported by the available evidence.

A strong and reliable failure of either property means that the
complete interaction does not satisfy the operational definition of
NORMAL.

Do not use a simple majority vote between the two properties.

Evidence that one property appears normal must not override a strong
and reliable failure of the other property.

In particular:

- Both participants speaking does not by itself prove NORMAL.
- Valid participation must not cancel reliable evidence that the
  participant timelines are not normally coordinated.
- Temporal compatibility must not cancel invalid participation.

============================================================
PARTICIPATION VALIDITY
============================================================

The field "speaks" is derived directly from the final filtered VAD turns:

- speaks = true means that the participant has at least one retained turn.
- speaks = false means that the participant has no retained turns during
  the entire 120-second interval.

Use the speaks fields.

For a normal spoken dyadic interaction, both participants are expected
to contribute speech during the complete interval.

Natural asymmetry is allowed:

- one participant may speak substantially more than the other,
- participants may have long listening periods,
- turn numbers and speaking durations do not need to be balanced.

However, the complete absence of retained speech from one participant
is not compatible with a normal two-person spoken interaction.

============================================================
LOCAL TEMPORAL FEATURES
============================================================

The signed strict A_end-to-B_start offsets are calculated as:

B_start minus A_end

Interpretation:

Negative value:

- Participant B starts shortly before Participant A finishes.

Value near zero:

- Participant B starts close to Participant A's turn boundary.

Positive value:

- Participant B starts after Participant A finishes.

The supplied local temporal evidence includes:

- the complete signed-offset list,
- number of valid offsets,
- mean,
- median,
- maximum,
- P75,
- P90,
- number of offsets above 1.5 seconds,
- percentage of offsets above 1.5 seconds.

Use the complete signed-offset distribution.

Do not decide from:

- one maximum value,
- one long pause,
- one negative value,
- or one isolated positive offset.

A NORMAL local temporal pattern is generally characterized by:

- repeated handoffs that are negative, near zero, or short positive,
- mean and median broadly compatible with the frozen NORMAL pattern,
- P75 and P90 broadly compatible with the frozen NORMAL pattern,
- offsets above 1.5 seconds being absent, uncommon, or isolated,
- no repeated and systematic pattern of substantially delayed handoffs.

Not every value must be negative or near zero.

A NORMAL interaction may contain occasional long pauses or unusual
handoffs.

However, repeated elevation across several parts of the offset
distribution is not equivalent to one isolated natural variation.

When the signed-offset list contains only one or two values, treat the
local temporal evidence as limited.

When the list is empty, the offset statistics are unavailable.
Do not invent missing evidence.

============================================================
FROZEN NORMAL LOCAL-TIMING REFERENCE
============================================================

The following statistics were calculated only from a frozen set of
separate NORMAL conversations:

Frozen NORMAL local temporal reference statistics:

- Number of signed offsets is around 5.76.
- Mean signed offset is around 0.30 seconds.
- Median signed offset is around 0.26 seconds.
- Maximum signed offset is around 1.05 seconds.
- P75 signed offset is around 0.59 seconds.
- P90 signed offset is around 0.83 seconds.
- Approximately 6.7% of offsets are above 1.5 seconds.

These values were calculated only from the frozen NORMAL reference conversations.
They are soft reference patterns and must not be treated as hard classification thresholds.

These statistics operationally describe the expected local temporal
pattern for NORMAL interactions in this experiment.

They are not independent hard thresholds.

A current case does not need to equal every average, and natural
variation around the reference pattern is expected.

However, these statistics are not optional background information.

Evaluate the current signed-offset distribution as a whole against the
frozen NORMAL reference.

A substantial and consistent departure across multiple reliable local
features means that the local temporal requirement for NORMAL is not
satisfied.

A multi-feature departure may include several of the following
appearing together:

- substantially elevated mean,
- substantially elevated median,
- substantially elevated P75 or P90,
- several offsets above 1.5 seconds,
- a substantially elevated percentage above 1.5 seconds,

One unusual feature alone is insufficient.

Several mutually supporting deviations are strong evidence.

============================================================
GLOBAL ALIGNMENT-SHIFT FEATURES
============================================================

The global alignment search evaluates hypothetical temporal corrections
to Participant B.

The global features have the following meanings:

- best_B_correction_shift_seconds is the hypothetical correction that
  produced the strongest bilateral turn-boundary alignment.

- A correction near zero means that little global temporal correction
  was preferred.

- A negative correction means that Participant B would align better if
  Participant B's timeline were moved earlier.

- estimated_B_lateness_seconds is the non-negative lateness implied by
  the best correction.

- alignment_score_gain_vs_zero measures how much the best correction
  improves alignment compared with applying no correction.

- best_num_bilateral_events indicates how many bilateral events support
  the selected correction.

- best_event_coverage indicates how much of the available interaction
  supports that estimate.

Do not use the correction value alone.

A large correction based on very few events or very low event coverage
is weak evidence.

A global estimate becomes more reliable when it is jointly supported by:

- a meaningful correction,
- meaningful estimated lateness,
- meaningful alignment improvement over zero shift,
- several bilateral events,
- sufficiently broad event coverage,
- and agreement with the local signed-offset pattern.

============================================================
FROZEN NORMAL GLOBAL-ALIGNMENT REFERENCE
============================================================

The following statistics were calculated only from the same frozen
NORMAL reference set:

Frozen NORMAL global alignment-shift reference statistics:

- Mean best B correction shift is around -0.23 seconds.
- Median best B correction shift is around 0.00 seconds.
- Mean estimated B lateness is around 0.50 seconds.
- Median estimated B lateness is around 0.00 seconds.
- Mean alignment score gain versus zero shift is around 0.023.
- Median alignment score gain versus zero shift is around 0.009.
- Mean number of bilateral alignment events is around 11.94.
- Mean bilateral event coverage is around 59.52%.

These values were calculated only from the frozen NORMAL reference conversations.
They describe typical global-alignment behavior in the frozen NORMAL set and are not hard classification thresholds.

These statistics operationally describe the expected global alignment
behavior for NORMAL interactions in this experiment.

They are not independent hard thresholds.

Natural variation is allowed, and one unusual global value does not
automatically exclude NORMAL.

For NORMAL temporal coordination, the global evidence is generally
expected to show:

- a preferred correction near the frozen NORMAL pattern,
- limited estimated lateness,
- limited improvement over applying no correction,
- or insufficient reliable evidence that a substantial correction
  is necessary.

The global temporal requirement for NORMAL is not satisfied when a
substantial correction is reliably supported by the evidence as a whole.

Evaluate together:

- correction direction and magnitude,
- estimated lateness,
- alignment-score gain over zero shift,
- number of bilateral events,
- event coverage,
- agreement with the local offset distribution.

A substantial correction with negligible gain, very few events, or
very low coverage is weak evidence.

A substantial correction with meaningful gain, sufficient bilateral
events, broad coverage, and matching local evidence is strong evidence
that the interaction does not follow the frozen NORMAL temporal pattern.

============================================================
JOINT TEMPORAL COORDINATION DECISION
============================================================

Local and global temporal evidence must be evaluated together.

Temporal coordination is a necessary and independent property of a
NORMAL interaction.

A case can fail the temporal requirement for NORMAL even when both
participants speak.

Treat the temporal dimension as compatible with NORMAL when:

- the signed-offset distribution is broadly consistent with the
  frozen NORMAL local pattern,
- long positive offsets are absent, uncommon, or isolated,
- elevated local values are not repeated across the distribution,
- the preferred global correction is near the frozen NORMAL pattern,
- or a larger correction is weakly supported by gain, events, or coverage.

Treat the temporal dimension as not compatible with NORMAL when:

- multiple local distribution statistics substantially depart from
  the frozen NORMAL pattern,
- delayed handoffs form a repeated rather than isolated pattern,
- and reliable global alignment evidence supports the same conclusion.

Do not require every temporal feature to depart from NORMAL.

Do not require every signed offset to be positive or large.

Do not allow one negative or near-zero offset to cancel a broader,
well-supported non-NORMAL temporal pattern.

When reliable local and global temporal evidence agree that the
interaction substantially departs from the frozen NORMAL pattern,
the complete case must be classified as ANOMALOUS.

Do not infer or report the cause or magnitude of the temporal failure.

============================================================
FINAL COMBINED DECISION
============================================================

Internally evaluate:

1. Participation validity
2. Temporal coordination

Then make one binary decision.

Classify as NORMAL only when the complete evidence is sufficiently
compatible with both required properties of a normal dyadic
interaction.

Classify as ANOMALOUS when either required property shows a strong,
reliable, and well-supported failure.

Do not require both properties to fail.

Do not allow strong evidence from one property to erase a reliable
failure in the other property.

In particular:

- Temporal compatibility must not cancel invalid participation.
- Speech from both participants must not cancel reliable temporal
  failure.

Do not classify as ANOMALOUS because of one isolated noisy measurement.

Do not classify as NORMAL merely because one evidence source appears
plausible.

Use the reliability and consistency of the evidence, not a simple count
of supportive features.

Return one final label even when some evidence is limited.

============================================================
CURRENT CASE
============================================================

Analysis duration:
{duration_seconds} seconds

PARTICIPANT A

Speaks:
{participant_A_speaks}

PARTICIPANT B

Speaks:
{participant_B_speaks}

LOCAL TEMPORAL FEATURES

{local_temporal_features}

GLOBAL ALIGNMENT-SHIFT FEATURES

{global_shift_features}

============================================================
STRUCTURED REASONING OUTPUT
============================================================

Preserve all evidence definitions, comparison rules, decision criteria,
and the final binary decision policy stated above.

Before selecting the final label, expose the following fixed structured
assessments.

These fields are diagnostic outputs only.

They must not introduce new evidence, new thresholds, a new voting rule,
or any change to the existing decision policy.

Use the available evidence exactly as instructed above.

Assessment meanings:

- participation_assessment:
  - VALID
  - INVALID

- local_temporal_assessment:
  - NORMAL
  - ANOMALOUS
  - LIMITED

- global_temporal_assessment:
  - NORMAL
  - ANOMALOUS
  - LIMITED

- temporal_assessment:
  - NORMAL
  - ANOMALOUS
  - LIMITED

- decisive_dimension:
  - PARTICIPATION
  - TEMPORAL
  - NONE

The local and global temporal assessments must reflect the reliability
rules already defined in the prompt.

The combined temporal assessment must be based on the existing temporal
decision policy.

The final label must follow the existing final decision policy exactly.

Do not infer or output an anomaly subtype.
Do not infer or output a delay magnitude.
Do not provide free-form reasoning.

============================================================
OUTPUT
============================================================

Return ONLY one valid JSON object with exactly this schema:

{{
  "participation_assessment": "VALID or INVALID",
  "local_temporal_assessment": "NORMAL or ANOMALOUS or LIMITED",
  "global_temporal_assessment": "NORMAL or ANOMALOUS or LIMITED",
  "temporal_assessment": "NORMAL or ANOMALOUS or LIMITED",
  "decisive_dimension": "PARTICIPATION or TEMPORAL or NONE",
  "label": "NORMAL or ANOMALOUS"
}}

Do not include confidence.
Do not include an anomaly type.
Do not include Markdown.
Do not include any text outside the JSON object.""".strip()


In [ ]:
# STEP S0.3 — PREPARE THE EXACT MANUALLY DEFINED EXPERIMENT

S0_NO_SEMANTIC_CONFIG = (
    prepare_manual_no_semantic_experiment(
        experiment_version=(
            "manual_ablation_s0_no_semantic_branch_"
            "speaks_offsets_all_global_no_overlap"
        ),
        experiment_title=(
            "S0 — No Semantic Branch — Manual Prompt — "
            "Speaks + Offsets + All Global — No Overlap"
        ),
        ablation_id=(
            "S0_NO_SEMANTIC_BRANCH_MANUAL_"
            "SPEAKS_OFFSETS_ALL_GLOBAL_NO_OVERLAP"
        ),
        manual_prompt_template=(
            S0_MANUAL_PROMPT_TEMPLATE
        ),
    )
)


S0 — No Semantic Branch — Manual Prompt — Speaks + Offsets + All Global — No Overlap — MANUAL CONFIGURATION READY
Ablation ID: S0_NO_SEMANTIC_BRANCH_MANUAL_SPEAKS_OFFSETS_ALL_GLOBAL_NO_OVERLAP
Semantic ablation mode: NO_SEMANTIC_BRANCH
Participant evidence: speaks only
Local evidence: complete offset distribution only
Global evidence: all global temporal features
Semantic evidence passed to model: False
Filtered turns passed to model: False
Overlap passed to model: False
Automatic prompt rewriting used: False
Schema keys: ['participation_assessment', 'local_temporal_assessment', 'global_temporal_assessment', 'temporal_assessment', 'decisive_dimension', 'label']
Baseline prompt SHA256: b6077b36aff2115441cfd76dbdec464758ee4c549233c92f4fe831aaca641d98
Manual prompt SHA256: 6429455c925bb18db0c4705f89384ff7283b41fdcf59aada82b4d5988a78cb6b
Prompt diff: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/manual_ablation_s0_no_semantic_branch_speaks_offsets_all_global_no_overlap

In [ ]:
# STEP S0.4 — REQUIRED PROMPT INSPECTION

S0_NO_SEMANTIC_INSPECTION = (
    inspect_manual_no_semantic_prompt(
        S0_NO_SEMANTIC_CONFIG,
        case_index=0,
    )
)


MANUAL PROMPT INSPECTION — S0 — No Semantic Branch — Manual Prompt — Speaks + Offsets + All Global — No Overlap
Inspection case ID: consolidation_lag_2sec_000
Gold label is intentionally NOT printed inside the model prompt.
Participant evidence supplied: speaks only.
Local evidence supplied: complete offset distribution only.
Global evidence supplied: all global temporal features.
Semantic evidence supplied: False
Filtered turns supplied: False
Overlap evidence supplied: False
Prompt definition method: manual complete template

EXACT ABLATED MODEL INPUT PAYLOAD
{
  "analysis_duration_seconds": 120.0,
  "participant_A": {
    "speaks": true
  },
  "participant_B": {
    "speaks": true
  },
  "local_temporal_features": {
    "signed_strict_offsets_seconds": [
      3.16,
      4.28
    ],
    "num_signed_strict_offsets": 2,
    "offset_mean_seconds": 3.72,
    "offset_median_seconds": 3.72,
    "offset_max_seconds": 4.28,
    "offset_p75_seconds": 4.0,
    "offset_p90_seconds": 4.17,
   

In [ ]:
# STEP S0.5 — RUN THE 400-CASE EXPERIMENT
# Set print_each_case_prompt=True only if all 400 prompts should be printed.

S0_NO_SEMANTIC_CACHE = (
    run_manual_no_semantic_experiment(
        S0_NO_SEMANTIC_CONFIG,
        print_each_case_prompt=False,
    )
)


Created cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/manual_ablation_s0_no_semantic_branch_speaks_offsets_all_global_no_overlap/predictions_cache.json


manual_ablation_s0_no_semantic_branch_speaks_offsets_all_global_no_overlap:   0%|          | 0/400 [00:00<?, ?…


S0 — No Semantic Branch — Manual Prompt — Speaks + Offsets + All Global — No Overlap — INFERENCE COMPLETE
Cached records: 400
Prediction cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/manual_ablation_s0_no_semantic_branch_speaks_offsets_all_global_no_overlap/predictions_cache.json


In [ ]:
# STEP S0.6 — EVALUATE

S0_NO_SEMANTIC_EVALUATION = (
    evaluate_reasoning_experiment(
        S0_NO_SEMANTIC_CONFIG
    )
)


S0 — No Semantic Branch — Manual Prompt — Speaks + Offsets + All Global — No Overlap — RESULTS
Total cases: 400
Valid predictions: 400
Invalid predictions: 0
Exact structured-schema rate: 1.0000
Accuracy: 0.8625
Balanced accuracy: 0.7317
ANOMALOUS precision: 0.8490
ANOMALOUS recall: 0.9933
ANOMALOUS F1: 0.9155
NORMAL recall / specificity: 0.4700
MCC: 0.6119
Matched source-group exact rate: 0.4600
Reasoning inconsistencies: 0

CONFUSION MATRIX


,Pred NORMAL,Pred ANOMALOUS
Gold NORMAL,47,53
Gold ANOMALOUS,2,298



CLASSIFICATION REPORT


,precision,recall,f1-score,support
NORMAL,0.959184,0.470000,0.630872,100.0000
ANOMALOUS,0.849003,0.993333,0.915515,300.0000
accuracy,0.862500,0.862500,0.862500,0.8625
macro avg,0.904093,0.731667,0.773194,400.0000
weighted avg,0.876548,0.862500,0.844354,400.0000



PER CASE FAMILY


...


PER EXACT CASE VARIANT


,case_variant,total_cases,valid_predictions,invalid_predictions,predicted_NORMAL,predicted_ANOMALOUS,correct_predictions,accuracy_on_valid,strict_accuracy_invalid_as_wrong,exact_schema_rate
0,lag_2sec,50,50,0,0,50,50,1.00,1.00,1.0
1,lag_3sec,50,50,0,1,49,49,0.98,0.98,1.0
2,normal,100,100,0,47,53,47,0.47,0.47,1.0
3,silent_partner,100,100,0,0,100,100,1.00,1.00,1.0
4,wrong_partner,100,100,0,1,99,99,0.99,0.99,1.0



ASSESSMENT DISTRIBUTIONS


,assessment_field,assessment_value,count
0,participation_assessment,VALID,300
1,participation_assessment,INVALID,100
2,local_temporal_assessment,ANOMALOUS,204
3,local_temporal_assessment,LIMITED,122
4,local_temporal_assessment,NORMAL,74
5,global_temporal_assessment,ANOMALOUS,228
6,global_temporal_assessment,LIMITED,123
7,global_temporal_assessment,NORMAL,49
8,temporal_assessment,ANOMALOUS,229
9,temporal_assessment,LIMITED,122



Saved cache: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/manual_ablation_s0_no_semantic_branch_speaks_offsets_all_global_no_overlap/predictions_cache.json
Saved prompt: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/manual_ablation_s0_no_semantic_branch_speaks_offsets_all_global_no_overlap/prompt_template.txt
Saved prompt diff: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/manual_ablation_s0_no_semantic_branch_speaks_offsets_all_global_no_overlap/manual_prompt_diff_vs_full_r1_baseline.txt
Saved predictions: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/manual_ablation_s0_no_semantic_branch_speaks_offsets_all_global_no_overlap/predictions_all_400.csv
Saved reasoning assessments: /content/drive/MyDrive/qwen_vad_turns_normal_vs_mixed_lag_1_2_3sec/manual_ablation_s0_no_semantic_branch_speaks_offsets_all_global_no_overlap/reasoning_assessments.csv
Saved errors: /content/drive/MyDrive/qwen_vad_turns_normal_

In [ ]:
import pandas as pd
from IPython.display import display


# ============================================================
# LOAD S0 — NO SEMANTIC BRANCH
# ============================================================

if "S0_NO_SEMANTIC_EVALUATION" in globals():

    s0_df = (
        S0_NO_SEMANTIC_EVALUATION[
            "results_df"
        ].copy()
    )

elif "S0_NO_SEMANTIC_CONFIG" in globals():

    s0_df = pd.read_csv(
        S0_NO_SEMANTIC_CONFIG[
            "paths"
        ][
            "predictions_csv"
        ]
    )

else:

    raise RuntimeError(
        "Run the S0_NO_SEMANTIC configuration "
        "and evaluation cells first."
    )


# ============================================================
# NORMALIZE TEXT FIELDS
# ============================================================

upper_columns = [
    "gold_label",
    "prediction",
    "participation_assessment",
    "local_temporal_assessment",
    "global_temporal_assessment",
    "temporal_assessment",
    "decisive_dimension",
]


for column in upper_columns:

    if column in s0_df.columns:

        s0_df[column] = (
            s0_df[column]
            .astype("string")
            .str.strip()
            .str.upper()
        )


for column in [
    "case_family",
    "case_variant",
]:

    if column in s0_df.columns:

        s0_df[column] = (
            s0_df[column]
            .astype("string")
            .str.strip()
            .str.lower()
            .str.replace(
                r"[\s\-]+",
                "_",
                regex=True,
            )
        )


# ============================================================
# VALID PREDICTION AND CORRECTNESS
# ============================================================

if "valid_prediction" not in s0_df.columns:

    s0_df["valid_prediction"] = (
        s0_df["prediction"].isin([
            "NORMAL",
            "ANOMALOUS",
        ])
    )


if "correct" not in s0_df.columns:

    s0_df["correct"] = (
        s0_df["valid_prediction"]
        &
        (
            s0_df["gold_label"]
            ==
            s0_df["prediction"]
        )
    )


# ============================================================
# EXPECTED STRUCTURED OUTPUT VALUES
# ============================================================

S0_ASSESSMENT_LEVELS = {

    "participation_assessment": [
        "VALID",
        "INVALID",
    ],

    "local_temporal_assessment": [
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    ],

    "global_temporal_assessment": [
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    ],

    "temporal_assessment": [
        "NORMAL",
        "ANOMALOUS",
        "LIMITED",
    ],

    "decisive_dimension": [
        "PARTICIPATION",
        "TEMPORAL",
        "NONE",
    ],
}


# ============================================================
# FREQUENCY TABLE
# ============================================================

def s0_frequency_table(
    subset,
    field,
    expected_values,
):

    if field not in subset.columns:

        return pd.DataFrame({
            "assessment_field": [field],
            "assessment_value": ["COLUMN_NOT_AVAILABLE"],
            "count": [0],
            "percentage": [0.0],
        })


    values = (
        subset[field]
        .fillna("MISSING")
        .astype(str)
        .str.strip()
        .str.upper()
    )


    ordered_values = list(
        dict.fromkeys(
            list(expected_values)
            +
            ["MISSING"]
        )
    )


    unexpected_values = [
        value
        for value in values.unique().tolist()
        if value not in ordered_values
    ]


    ordered_values.extend(
        sorted(unexpected_values)
    )


    counts = (
        values
        .value_counts(dropna=False)
        .reindex(
            ordered_values,
            fill_value=0,
        )
    )


    table = pd.DataFrame({
        "assessment_field": field,
        "assessment_value": counts.index,
        "count": counts.values,
    })


    if len(subset) > 0:

        table["percentage"] = (
            100.0
            *
            table["count"]
            /
            len(subset)
        ).round(2)

    else:

        table["percentage"] = 0.0


    return table


# ============================================================
# SAFE CROSSTAB
# ============================================================

def display_s0_crosstab(
    subset,
    row_field,
    column_field,
    title,
):

    print(
        f"\n{title}"
    )


    if (
        row_field not in subset.columns
        or
        column_field not in subset.columns
    ):

        print(
            "Required columns are unavailable."
        )

        return


    display(
        pd.crosstab(
            subset[row_field].fillna("MISSING"),
            subset[column_field].fillna("MISSING"),
            margins=True,
        )
    )


# ============================================================
# SHOW INDIVIDUAL CASES
# ============================================================

def display_s0_individual_cases(
    subset,
):

    preferred_columns = [
        "case_id",
        "case_family",
        "case_variant",
        "gold_label",
        "prediction",
        "participation_assessment",
        "local_temporal_assessment",
        "global_temporal_assessment",
        "temporal_assessment",
        "decisive_dimension",
        "correct",
    ]


    available_columns = [
        column
        for column in preferred_columns
        if column in subset.columns
    ]


    if not available_columns:

        print(
            "No case-level display columns are available."
        )

        return


    print(
        "\nINDIVIDUAL CASES"
    )


    case_table = (
        subset[available_columns]
        .sort_values(
            by=[
                column
                for column in [
                    "case_variant",
                    "case_id",
                ]
                if column in available_columns
            ]
        )
        .reset_index(drop=True)
    )


    display(
        case_table
    )


# ============================================================
# COMPLETE SUBSET INSPECTION
# ============================================================

def inspect_s0_subset(
    subset,
    title,
    show_individual_cases=True,
):

    subset = subset.copy()


    print(
        "\n"
        +
        "=" * 100
    )

    print(
        title
    )

    print(
        "=" * 100
    )


    if subset.empty:

        print(
            "No cases found in this subset."
        )

        return subset


    overview = pd.DataFrame([{

        "cases": len(subset),

        "gold_NORMAL": int(
            (
                subset["gold_label"]
                ==
                "NORMAL"
            ).sum()
        ),

        "gold_ANOMALOUS": int(
            (
                subset["gold_label"]
                ==
                "ANOMALOUS"
            ).sum()
        ),

        "pred_NORMAL": int(
            (
                subset["prediction"]
                ==
                "NORMAL"
            ).sum()
        ),

        "pred_ANOMALOUS": int(
            (
                subset["prediction"]
                ==
                "ANOMALOUS"
            ).sum()
        ),

        "invalid_predictions": int(
            (
                ~subset["valid_prediction"]
            ).sum()
        ),

        "correct": int(
            subset["correct"].sum()
        ),

        "accuracy_percent": round(
            100.0
            *
            subset["correct"].mean(),
            2,
        ),
    }])


    print(
        "\nSUBSET OVERVIEW"
    )

    display(
        overview
    )


    if "case_variant" in subset.columns:

        print(
            "\nCASE VARIANT BREAKDOWN"
        )


        variant_counts = (
            subset["case_variant"]
            .fillna("MISSING")
            .value_counts()
            .rename_axis("case_variant")
            .reset_index(name="count")
        )


        variant_counts["percentage"] = (
            100.0
            *
            variant_counts["count"]
            /
            len(subset)
        ).round(2)


        display(
            variant_counts
        )


    print(
        "\nASSESSMENT BREAKDOWN"
    )


    assessment_table = pd.concat(
        [
            s0_frequency_table(
                subset=subset,
                field=field,
                expected_values=expected_values,
            )
            for field, expected_values
            in S0_ASSESSMENT_LEVELS.items()
        ],
        ignore_index=True,
    )


    display(
        assessment_table
    )


    display_s0_crosstab(
        subset,
        "participation_assessment",
        "decisive_dimension",
        (
            "PARTICIPATION ASSESSMENT "
            "× DECISIVE DIMENSION"
        ),
    )


    display_s0_crosstab(
        subset,
        "local_temporal_assessment",
        "global_temporal_assessment",
        (
            "LOCAL × GLOBAL "
            "TEMPORAL ASSESSMENT"
        ),
    )


    display_s0_crosstab(
        subset,
        "local_temporal_assessment",
        "temporal_assessment",
        (
            "LOCAL TEMPORAL ASSESSMENT "
            "× COMBINED TEMPORAL ASSESSMENT"
        ),
    )


    display_s0_crosstab(
        subset,
        "global_temporal_assessment",
        "temporal_assessment",
        (
            "GLOBAL TEMPORAL ASSESSMENT "
            "× COMBINED TEMPORAL ASSESSMENT"
        ),
    )


    display_s0_crosstab(
        subset,
        "temporal_assessment",
        "decisive_dimension",
        (
            "TEMPORAL ASSESSMENT "
            "× DECISIVE DIMENSION"
        ),
    )


    if show_individual_cases:

        display_s0_individual_cases(
            subset
        )


    return subset


# ============================================================
# FAMILY MASK
# ============================================================

def get_s0_family_mask(
    dataframe,
    family,
):

    values = (
        dataframe["case_family"]
        .fillna("")
        .astype(str)
        .str.lower()
    )


    if family == "normal":

        return values.isin([
            "normal",
            "normals",
        ])


    if family == "lag":

        return (
            values.isin([
                "lag",
                "lags",
            ])
            |
            values.str.contains(
                r"(?:^|_)lag(?:$|_)",
                regex=True,
            )
        )


    if family == "wrong_partner":

        return (
            values.isin([
                "wrong_partner",
                "wrong_partners",
            ])
            |
            (
                values.str.contains(
                    "wrong",
                    regex=False,
                )
                &
                values.str.contains(
                    "partner",
                    regex=False,
                )
            )
        )


    if family == "silent_partner":

        return (
            values.isin([
                "silent_partner",
                "silent_partners",
            ])
            |
            (
                values.str.contains(
                    "silent",
                    regex=False,
                )
                &
                values.str.contains(
                    "partner",
                    regex=False,
                )
            )
        )


    raise ValueError(
        f"Unknown family: {family}"
    )


# ============================================================
# CREATE OUTCOME SUBSET
# ============================================================

def get_s0_outcome_subset(
    family,
    outcome,
):

    family_mask = get_s0_family_mask(
        dataframe=s0_df,
        family=family,
    )


    if family == "normal":

        gold_label = "NORMAL"

        if outcome == "correct":

            prediction = "NORMAL"

        elif outcome == "missed":

            prediction = "ANOMALOUS"

        else:

            raise ValueError(
                "outcome must be 'correct' or 'missed'."
            )

    else:

        gold_label = "ANOMALOUS"

        if outcome == "correct":

            prediction = "ANOMALOUS"

        elif outcome == "missed":

            prediction = "NORMAL"

        else:

            raise ValueError(
                "outcome must be 'correct' or 'missed'."
            )


    return s0_df[
        family_mask
        &
        (
            s0_df["gold_label"]
            ==
            gold_label
        )
        &
        (
            s0_df["prediction"]
            ==
            prediction
        )
    ].copy()


# ============================================================
# INITIAL CHECK
# ============================================================

print(
    "S0 cases loaded:",
    len(s0_df),
)


print(
    "\nCASE FAMILY COUNTS"
)

display(
    s0_df["case_family"]
    .fillna("MISSING")
    .value_counts()
    .rename_axis("case_family")
    .reset_index(name="count")
)


print(
    "\nCASE FAMILY × PREDICTION"
)

display(
    pd.crosstab(
        s0_df["case_family"],
        s0_df["prediction"],
        margins=True,
    )
)

S0 cases loaded: 400

CASE FAMILY COUNTS


,case_family,count
0,normal,100
1,wrong_partner,100
2,lag,100
3,silent_partner,100



CASE FAMILY × PREDICTION


prediction,ANOMALOUS,NORMAL,All
case_family,,,
lag,99,1,100
normal,53,47,100
silent_partner,100,0,100
wrong_partner,99,1,100
All,351,49,400


In [ ]:
# ============================================================
# LAG — CORRECTLY DETECTED
# ============================================================

s0_lag_correct = get_s0_outcome_subset(
    family="lag",
    outcome="correct",
)

inspect_s0_subset(
    s0_lag_correct,
    (
        "S0 — LAG CASES CORRECTLY "
        "DETECTED AS ANOMALOUS"
    ),
)


# ============================================================
# LAG — MISSED
# ============================================================

s0_lag_missed = get_s0_outcome_subset(
    family="lag",
    outcome="missed",
)

inspect_s0_subset(
    s0_lag_missed,
    (
        "S0 — LAG CASES NOT DETECTED "
        "(PREDICTED NORMAL)"
    ),
)


# ============================================================
# WRONG PARTNER — CORRECTLY DETECTED
# ============================================================

s0_wrong_partner_correct = get_s0_outcome_subset(
    family="wrong_partner",
    outcome="correct",
)

inspect_s0_subset(
    s0_wrong_partner_correct,
    (
        "S0 — WRONG-PARTNER CASES CORRECTLY "
        "DETECTED AS ANOMALOUS"
    ),
)


# ============================================================
# WRONG PARTNER — MISSED
# ============================================================

s0_wrong_partner_missed = get_s0_outcome_subset(
    family="wrong_partner",
    outcome="missed",
)

inspect_s0_subset(
    s0_wrong_partner_missed,
    (
        "S0 — WRONG-PARTNER CASES NOT DETECTED "
        "(PREDICTED NORMAL)"
    ),
)


# ============================================================
# NORMAL — CORRECTLY DETECTED
# ============================================================

s0_normal_correct = get_s0_outcome_subset(
    family="normal",
    outcome="correct",
)

inspect_s0_subset(
    s0_normal_correct,
    (
        "S0 — NORMAL CASES CORRECTLY "
        "PREDICTED AS NORMAL"
    ),
)


# ============================================================
# NORMAL — FALSELY FLAGGED
# ============================================================

s0_normal_missed = get_s0_outcome_subset(
    family="normal",
    outcome="missed",
)

inspect_s0_subset(
    s0_normal_missed,
    (
        "S0 — NORMAL CASES INCORRECTLY "
        "PREDICTED AS ANOMALOUS"
    ),
)


# ============================================================
# SILENT PARTNER — CORRECTLY DETECTED
# ============================================================

s0_silent_partner_correct = get_s0_outcome_subset(
    family="silent_partner",
    outcome="correct",
)

inspect_s0_subset(
    s0_silent_partner_correct,
    (
        "S0 — SILENT-PARTNER CASES CORRECTLY "
        "DETECTED AS ANOMALOUS"
    ),
)


# ============================================================
# SILENT PARTNER — MISSED
# ============================================================

s0_silent_partner_missed = get_s0_outcome_subset(
    family="silent_partner",
    outcome="missed",
)

inspect_s0_subset(
    s0_silent_partner_missed,
    (
        "S0 — SILENT-PARTNER CASES NOT DETECTED "
        "(PREDICTED NORMAL)"
    ),
)


S0 — LAG CASES CORRECTLY DETECTED AS ANOMALOUS

SUBSET OVERVIEW


,cases,gold_NORMAL,gold_ANOMALOUS,pred_NORMAL,pred_ANOMALOUS,invalid_predictions,correct,accuracy_percent
0,99,0,99,0,99,0,99,100.0



CASE VARIANT BREAKDOWN


,case_variant,count,percentage
0,lag_2sec,50,50.51
1,lag_3sec,49,49.49



ASSESSMENT BREAKDOWN


...

decisive_dimension,PARTICIPATION,All
temporal_assessment,,
LIMITED,100,100
All,100,100



INDIVIDUAL CASES


,case_id,case_family,case_variant,gold_label,prediction,participation_assessment,local_temporal_assessment,global_temporal_assessment,temporal_assessment,decisive_dimension,correct
0,consolidation_silent_partner_000,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,LIMITED,LIMITED,LIMITED,PARTICIPATION,True
1,consolidation_silent_partner_001,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,LIMITED,LIMITED,LIMITED,PARTICIPATION,True
2,consolidation_silent_partner_002,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,LIMITED,LIMITED,LIMITED,PARTICIPATION,True
3,consolidation_silent_partner_003,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,LIMITED,LIMITED,LIMITED,PARTICIPATION,True
4,consolidation_silent_partner_004,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,LIMITED,LIMITED,LIMITED,PARTICIPATION,True
...,...,...,...,...,...,...,...,...,...,...,...
95,consolidation_silent_partner_095,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,LIMITED,LIMITED,LIMITED,PARTICIPATION,True
96,consolidation_silent_partner_096,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,LIMITED,LIMITED,LIMITED,PARTICIPATION,True
97,consolidation_silent_partner_097,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,LIMITED,LIMITED,LIMITED,PARTICIPATION,True
98,consolidation_silent_partner_098,silent_partner,silent_partner,ANOMALOUS,ANOMALOUS,INVALID,LIMITED,LIMITED,LIMITED,PARTICIPATION,True



S0 — SILENT-PARTNER CASES NOT DETECTED (PREDICTED NORMAL)
No cases found in this subset.


,case_id,source_group_id,case_family,case_variant,gold_label,prediction,participation_assessment,local_temporal_assessment,global_temporal_assessment,temporal_assessment,...,input_token_count,elapsed_seconds,raw_output,generation_error,valid_prediction,correct,normal_with_invalid_participation,normal_with_anomalous_temporal,normal_with_incompatible_semantics,reasoning_inconsistency


# Optional: disconnect the Colab runtime

Run this only after the experiment has finished or after you have intentionally stopped it. Every completed case is checkpointed in Google Drive.


In [ ]:
from google.colab import runtime

runtime.unassign()
